In [13]:
# --- CELL 1: CONFIG ---
BASE_MODEL = "microsoft/deberta-v3-large"
SEED = 42
EPOCHS_FROZEN = 1
EPOCHS_FINETUNE = 1
BATCH_BUNDLES = 1          # Reduced to 1 to fit safely in T4 VRAM
ACCUM_STEPS = 8            # Increased to 8 to keep the effective batch size identical
MAX_LEN = 512
HEAD_LR = 5e-4
BASE_LR = 1e-5
CONSISTENCY_WEIGHT = 1.0
SMOKE_BUNDLES = 50
OUT_NAME = "myna-v5-prototype"
print("BASE:", BASE_MODEL, "| epochs:", EPOCHS_FROZEN, "frozen +", EPOCHS_FINETUNE, "fine-tune")

BASE: microsoft/deberta-v3-large | epochs: 1 frozen + 1 fine-tune


In [14]:
# --- CELL 2: GPU + Drive + AMP Dtype Setup ---
import os, gc, torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    major, _ = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
    print("Using AMP_DTYPE:", AMP_DTYPE)
else:
    AMP_DTYPE = torch.float32
    print("STOP: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.")

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = f"/content/drive/MyDrive/{OUT_NAME}"
except Exception as e:
    print("Drive fallback to /content:", e)
    OUT_DIR = f"/content/{OUT_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

PyTorch: 2.11.0+cu130 | CUDA available: True
GPU: Tesla T4
Using AMP_DTYPE: torch.float16
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR: /content/drive/MyDrive/myna-v5-prototype


In [10]:
# --- CELL 3: Clone Myna + Install ---
import os
ROOT_DIR = '/content' if os.path.isdir('/content') else os.getcwd()
REPO_DIR = os.path.join(ROOT_DIR, 'myna-repo')
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull --ff-only 2>&1 | tail -2
else:
    os.chdir(ROOT_DIR)
    !git clone -q https://github.com/aashish254/myna.git {REPO_DIR} 2>&1 | tail -2
    os.chdir(REPO_DIR)

!pip install -q -e ".[bench]" sentencepiece tiktoken
print("Working directory:", os.getcwd())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for myna-engine (pyproject.toml) ... done
Working directory: /content/myna-repo


In [11]:
# --- CELL 4: Multi-Question Document Bundle Loader ---
import json, hashlib

def load_bundles(path):
    bundles = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            state = d['state']
            src = d['_meta']['source']
            questions = []
            for qname, q in d['questions'].items():
                qtype = q['type']
                if qtype == 'choice':
                    opts = list(q['criteria'].keys())
                    gold = opts.index(q['label']) if q['label'] in opts else 0
                    questions.append({
                        'qname': qname, 'type': 'choice',
                        'instructions': q['instructions'],
                        'options': opts, 'descs': [q['criteria'][o] for o in opts],
                        'gold': gold
                    })
                elif qtype == 'noul':
                    gold = 1.0 if q['label'] else 0.0
                    questions.append({
                        'qname': qname, 'type': 'noul',
                        'instructions': q['instructions'],
                        'options': ['false', 'true'],
                        'descs': ['statement is false', 'statement is true'],
                        'gold': gold
                    })
                elif qtype == 'score':
                    levels = q['criteria']
                    gold = int(q['label']) if isinstance(q['label'], (int, float)) else 0
                    questions.append({
                        'qname': qname, 'type': 'score',
                        'instructions': q['instructions'],
                        'options': [str(i) for i in range(len(levels))],
                        'descs': levels,
                        'gold': gold
                    })
            if questions:
                bundles.append({'source': src, 'state': state, 'questions': questions})
    return bundles

train_bundles = load_bundles('data/decision-v2-pilot/train.jsonl')
dev_bundles = load_bundles('data/decision-v2-pilot/development.jsonl')
test_bundles = load_bundles('data/decision-v2-pilot/test.jsonl')

print(f"train bundles: {len(train_bundles)} | dev: {len(dev_bundles)} | test: {len(test_bundles)}")

train bundles: 57904 | dev: 1176 | test: 1176


In [ ]:
# --- CELL 5: Load Tokenizer & Base Model ---
import torch
from transformers import AutoTokenizer, AutoModel

try:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
except Exception:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

base = AutoModel.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
print("base hidden size:", base.config.hidden_size)

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  874MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


base hidden size: 1024


In [ ]:
# --- CELL 6: Example Builder ---
MASK_ID = tok.mask_token_id
CLS_ID = tok.cls_token_id
SEP_ID = tok.sep_token_id
PAD_ID = tok.pad_token_id

def text_of(x):
    if isinstance(x, str): return x
    if x is None: return ''
    if isinstance(x, dict): return ' | '.join(f"{k}: {text_of(v)}" for k, v in x.items())
    if isinstance(x, list): return ' '.join(text_of(v) for v in x)
    return str(x)

def build_bundle_examples(bundle):
    state_text = text_of(bundle['state'])
    encoded_qs = []
    for q in bundle['questions']:
        n_opt = len(q['options'])
        max_opt_toks = max(2, (MAX_LEN - 96) // max(1, n_opt) - 2)

        opt_chunks = []
        for label, desc in zip(q['options'], q['descs']):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d and max_opt_toks > 4:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids'][:max_opt_toks]
            if not opt_ids: opt_ids = [tok.unk_token_id]
            opt_chunks.append(opt_ids + [MASK_ID, SEP_ID])

        prefix = f"Task ({bundle['source']} / {q['qname']}): {text_of(q['instructions'])} | Context: "
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids'][:64]
        state_ids = tok(state_text, add_special_tokens=False)['input_ids'][: (MAX_LEN - 64 - sum(len(c) for c in opt_chunks))]

        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID]
        mask_pos = []
        for chunk in opt_chunks:
            if len(ids) + len(chunk) > MAX_LEN: break
            ids += chunk
            mask_pos.append(len(ids) - 2)

        encoded_qs.append({
            'qname': q['qname'],
            'type': q['type'],
            'input_ids': ids,
            'mask_pos': mask_pos,
            'gold': q['gold'],
            'n_opt': len(mask_pos)
        })
    return {'source': bundle['source'], 'questions': encoded_qs}

print("Sample built bundle test:", len(build_bundle_examples(train_bundles[0])['questions']))

Sample built bundle test: 3


In [ ]:
# --- CELL 7: Learned Bilinear Question-Graph Head (Fixed Shapes) ---
import torch.nn as nn
import torch.nn.functional as F

class QuestionGraphHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, base_model, input_ids, attn, mask_pos, opt_mask):
        B, M, L = input_ids.shape
        # Flatten batch and question dimensions to process through DeBERTa safely
        flat_ids = input_ids.view(B * M, L)
        flat_attn = attn.view(B * M, L)

        out = base_model(input_ids=flat_ids, attention_mask=flat_attn).last_hidden_state

        _, _, O = mask_pos.shape
        flat_mpos = mask_pos.view(B * M, O)

        idx = torch.arange(B * M, device=input_ids.device).unsqueeze(1)
        h_options = out[idx, flat_mpos] # (B*M, O, hidden)

        logits = self.scorer(h_options).squeeze(-1).float()
        logits = logits.view(B, M, O)

        omask_flat = opt_mask.view(B, M, O)
        logits = logits.masked_fill(omask_flat == 0, -1e9)
        return logits

def bundle_collate(bundle_list):
    max_q = max(len(b['questions']) for b in bundle_list)
    max_len = max(max(len(q['input_ids']) for q in b['questions']) for b in bundle_list)
    max_opt = max(max(len(q['mask_pos']) for q in b['questions']) for b in bundle_list)
    B = len(bundle_list)

    ids = torch.full((B, max_q, max_len), PAD_ID, dtype=torch.long)
    attn = torch.zeros((B, max_q, max_len), dtype=torch.long)
    mpos = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    omask = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    golds = []

    for i, b in enumerate(bundle_list):
        g_list = []
        for j, q in enumerate(b['questions']):
            ids[i, j, :len(q['input_ids'])] = torch.tensor(q['input_ids'])
            attn[i, j, :len(q['input_ids'])] = 1
            mpos[i, j, :len(q['mask_pos'])] = torch.tensor(q['mask_pos'])
            omask[i, j, :len(q['mask_pos'])] = 1
            g_list.append(q['gold'])
        golds.append(g_list)
    return ids, attn, mpos, omask, golds, bundle_list

torch.manual_seed(SEED)
model = QuestionGraphHead(base.config.hidden_size)
print("Question Graph Head initialized successfully with fixed tensor flattening.")

Question Graph Head initialized successfully with fixed tensor flattening.


In [ ]:
# --- CELL 8: SMOKE TEST (Fixed Cross Entropy Dtype) ---
from torch.utils.data import DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base.to(device)
model.to(device)

smoke_data = [build_bundle_examples(x) for x in train_bundles[:SMOKE_BUNDLES]]
smoke_dl = DataLoader(smoke_data, batch_size=2, shuffle=True, collate_fn=bundle_collate)

model.train()
opt = torch.optim.AdamW(model.parameters(), lr=HEAD_LR)

for ids, attn, mpos, omask, golds, _ in smoke_dl:
    ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)

    # Forward pass
    logits = model(base, ids, attn, mpos, omask) # (B, M, O)

    loss = 0.0
    for j in range(logits.size(1)):
        q_logits = logits[:, j, :].float() # Ensure float32 for cross_entropy
        # CRITICAL FIX: target must be torch.long, not float/int32
        q_golds = torch.tensor([g[j] for g in golds], dtype=torch.long, device=device)

        loss += F.cross_entropy(q_logits, q_golds)

    loss.backward()
    opt.step()
    opt.zero_grad()
    print("Smoke step loss successfully computed:", loss.item())
    break

print("SMOKE TEST PASSED.")

Smoke step loss successfully computed: 2.75095796585083
SMOKE TEST PASSED.


In [ ]:
# --- CELL 9: RESUME AND RUN EPOCH 2 WITH GRADIENT CHECKPOINTING ---
import math, time, random
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

random.seed(SEED)
torch.manual_seed(SEED)

train_built = [build_bundle_examples(x) for x in train_bundles]
dev_built = [build_bundle_examples(x) for x in dev_bundles[:200]]
print(f"Training bundles built: {len(train_built)} | Dev bundles: {len(dev_built)}")

@torch.no_grad()
def evaluate_dev_bundles():
    model.eval()
    dl_dev = DataLoader(dev_built, batch_size=2, shuffle=False, collate_fn=bundle_collate)
    correct, total = 0, 0
    for ids, attn, mpos, omask, golds, raw_bundles in dl_dev:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_bundles):
            n_qs = len(b['questions'])
            for j in range(n_qs):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                correct += int(pred == gold)
                total += 1
    return correct / max(total, 1)

best_ckpt_path = f"{OUT_DIR}/myna-v5-best.pt"
best_dev_acc = 0.50  # From Epoch 1 completion

# Load the saved Epoch 1 checkpoint so we skip retraining it
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(ckpt['state'])
    best_dev_acc = ckpt.get('dev_acc', 0.50)
    print(f"Successfully loaded best Epoch 1 checkpoint from {best_ckpt_path} (dev_acc={best_dev_acc:.4f})")
else:
    raise FileNotFoundError(f"Checkpoint not found at {best_ckpt_path}! Make sure Drive is mounted.")

def run_finetune_epoch(epoch):
    global best_dev_acc
    # Unfreeze base model and enable gradient checkpointing to save VRAM
    for p in base.parameters():
        p.requires_grad = True
    if hasattr(base, "gradient_checkpointing_enable"):
        base.gradient_checkpointing_enable()
        print("Gradient checkpointing enabled on base model.")

    params = [
        {'params': base.parameters(), 'lr': BASE_LR},
        {'params': model.parameters(), 'lr': HEAD_LR * 0.5}
    ]

    opt = torch.optim.AdamW(params, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))
    dl = DataLoader(train_built, batch_size=BATCH_BUNDLES, shuffle=True, collate_fn=bundle_collate)
    steps = math.ceil(len(dl) / ACCUM_STEPS)

    sched = get_cosine_schedule_with_warmup(opt, int(0.1 * steps), steps)

    model.train()
    opt.zero_grad()
    t0 = time.time()
    tot, n = 0.0, 0

    for i, (ids, attn, mpos, omask, golds, raw_bundles) in enumerate(dl):
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)

        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)
            loss = 0.0
            for b_idx, b in enumerate(raw_bundles):
                n_qs = len(b['questions'])
                for j in range(n_qs):
                    q_logits = logits[b_idx, j, :].float()
                    q_gold = torch.tensor(golds[b_idx][j], dtype=torch.long, device=device)
                    loss += F.cross_entropy(q_logits.unsqueeze(0), q_gold.unsqueeze(0))
            loss = loss / ACCUM_STEPS

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"WARNING: NaN/Inf loss at epoch {epoch} batch {i+1}")
            opt.zero_grad()
            continue

        scaler.scale(loss).backward()
        tot += loss.item() * ACCUM_STEPS
        n += 1

        if (i + 1) % ACCUM_STEPS == 0 or i == len(dl) - 1:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(opt)
            scaler.update()

            sched.step()
            opt.zero_grad()

        if (i + 1) % 400 == 0:
            print(f"Epoch {epoch} (finetune) batch {i+1}/{len(dl)} loss {tot/max(n,1):.4f} elapsed {time.time()-t0:.0f}s")

    dev_acc = evaluate_dev_bundles()
    print(f"Epoch {epoch} fine-tune done | Mean train loss: {tot/max(n,1):.4f} | Dev Bundle Accuracy: {dev_acc:.4f}")

    if dev_acc >= best_dev_acc:
        best_dev_acc = dev_acc
        torch.save({'state': model.state_dict(), 'dev_acc': best_dev_acc}, best_ckpt_path)
        print(f"  -> New best dev accuracy saved to {best_ckpt_path}")

    gc.collect()
    torch.cuda.empty_cache()

# Run Epoch 2 safely with gradient checkpointing
run_finetune_epoch(2)
print("Training complete!")

Training bundles built: 57904 | Dev bundles: 200
Successfully loaded best Epoch 1 checkpoint from /content/drive/MyDrive/myna-v5-prototype/myna-v5-best.pt (dev_acc=0.5000)
Gradient checkpointing enabled on base model.
Epoch 2 (finetune) batch 400/57904 loss 1.2460 elapsed 76s
Epoch 2 (finetune) batch 800/57904 loss 1.2298 elapsed 152s
Epoch 2 (finetune) batch 1200/57904 loss 1.2532 elapsed 228s
Epoch 2 (finetune) batch 1600/57904 loss 1.2833 elapsed 304s
Epoch 2 (finetune) batch 2000/57904 loss 1.2878 elapsed 379s
Epoch 2 (finetune) batch 2400/57904 loss 1.2702 elapsed 455s
Epoch 2 (finetune) batch 2800/57904 loss 1.2629 elapsed 531s
Epoch 2 (finetune) batch 3200/57904 loss 1.2598 elapsed 608s
Epoch 2 (finetune) batch 3600/57904 loss 1.2413 elapsed 684s
Epoch 2 (finetune) batch 4000/57904 loss 1.2225 elapsed 761s
Epoch 2 (finetune) batch 4400/57904 loss 1.2086 elapsed 837s
Epoch 2 (finetune) batch 4800/57904 loss 1.1974 elapsed 913s
Epoch 2 (finetune) batch 5200/57904 loss 1.1856 elaps

In [ ]:
# --- CELL 10: EVALUATION ON TEST BUNDLES ---
test_built = [build_bundle_examples(x) for x in test_bundles]

@torch.no_grad()
def evaluate_test():
    model.eval()
    dl_test = DataLoader(test_built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    correct, total = 0, 0
    primitive_stats = {'choice': [0, 0], 'score': [0, 0], 'noul': [0, 0]}

    for ids, attn, mpos, omask, golds, raw_bundles in dl_test:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_bundles):
            for j, q in enumerate(b['questions']):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                ptype = q['type']

                is_correct = int(pred == gold)
                correct += is_correct
                total += 1
                if ptype in primitive_stats:
                    primitive_stats[ptype][0] += is_correct
                    primitive_stats[ptype][1] += 1

    overall_acc = correct / max(total, 1)
    print(f"=== TEST EVALUATION REPORT ===")
    print(f"Overall Multi-Primitive Accuracy: {correct}/{total} = {overall_acc:.4f}")
    for ptype, (c, t) in primitive_stats.items():
        if t > 0:
            print(f"  - {ptype}: {c}/{t} = {c/t:.4f}")
    return overall_acc, primitive_stats

test_acc, stats = evaluate_test()

=== TEST EVALUATION REPORT ===
Overall Multi-Primitive Accuracy: 1162/1440 = 0.8069
  - choice: 541/696 = 0.7773
  - score: 152/240 = 0.6333
  - noul: 469/504 = 0.9306


In [ ]:
# --- CELL 11: WITNESS JSON GENERATION ---
import datetime

witness = {
    'model_label': 'myna-v5-prototype (DeBERTa-v3-large + Question-Graph Multi-Primitive Head)',
    'base_model': BASE_MODEL,
    'seed': SEED,
    'best_dev_acc': best_dev_acc,
    'test_accuracy': test_acc,
    'timestamp': datetime.datetime.now().isoformat(),
    'notes': 'Evaluated across all 3 primitives (choice, score, noul) grouped by document bundles without hardcoded argmax hacks.'
}

witness_path = f"{OUT_DIR}/witness-v5.json"
with open(witness_path, 'w', encoding='utf-8') as f:
    json.dump(witness, f, indent=2)

print("Witness saved successfully to:", witness_path)
print(json.dumps(witness, indent=2))

Witness saved successfully to: /content/drive/MyDrive/myna-v5-prototype/witness-v5.json
{
  "model_label": "myna-v5-prototype (DeBERTa-v3-large + Question-Graph Multi-Primitive Head)",
  "base_model": "microsoft/deberta-v3-large",
  "seed": 42,
  "best_dev_acc": 0.65,
  "test_accuracy": 0.8069444444444445,
  "timestamp": "2026-10-31T22:22:59.832504",
  "notes": "Evaluated across all 3 primitives (choice, score, noul) grouped by document bundles without hardcoded argmax hacks."
}


In [ ]:
# --- FIXED CELL 12: SAVE FULL BACKBONE + COMPUTE PAPER AUDIT ---
from collections import defaultdict

# 1. Save the fine-tuned DeBERTa-v3-large backbone using the active tokenizer reference
try:
    base.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    if 'tokenizer' in globals():
        tokenizer.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    elif 'tok' in globals():
        tok.save_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
    print("Saved fine-tuned DeBERTa backbone and tokenizer to Google Drive!")
except Exception as e:
    print(f"Note on save: {e}")

@torch.no_grad()
def full_paper_audit(bundles_raw, split_name="test"):
    model.eval()
    built = [build_bundle_examples(x) for x in bundles_raw]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)

    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q_bundles, contradicted_bundles = 0, 0
    cell_stats = defaultdict(lambda: [0, 0])
    prim_stats = defaultdict(lambda: [0, 0])

    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask)

        for b_idx, b in enumerate(raw_b):
            qs = b['questions']
            b_all_correct = True
            preds_by_type = {}

            for j, q in enumerate(qs):
                pred = logits[b_idx, j, :].argmax(dim=-1).item()
                gold = golds[b_idx][j]
                ok = int(pred == gold)

                correct_q += ok
                total_q += 1
                if not ok:
                    b_all_correct = False

                ptype = q['type']
                prim_stats[ptype][0] += ok
                prim_stats[ptype][1] += 1

                src = b.get('source', 'unknown')
                qid = q.get('qid', q.get('id', ptype))
                cell_stats[(src, qid)][0] += ok
                cell_stats[(src, qid)][1] += 1
                preds_by_type[ptype] = (pred, q)

            total_bundles += 1
            if b_all_correct:
                exact_bundles += 1

            if len(qs) > 1:
                multi_q_bundles += 1
                if 'choice' in preds_by_type and 'noul' in preds_by_type:
                    c_pred, c_q = preds_by_type['choice']
                    n_pred, n_q = preds_by_type['noul']
                    c_opts = list(c_q.get('criteria', {}).keys()) if isinstance(c_q.get('criteria'), dict) else []
                    if c_pred < len(c_opts):
                        chosen_label = str(c_opts[c_pred]).lower()
                        n_inst = str(n_q.get('instructions', '')).lower()
                        if chosen_label in n_inst and n_pred == 0:
                            contradicted_bundles += 1

    overall_acc = correct_q / max(total_q, 1)
    bundle_exact_acc = exact_bundles / max(total_bundles, 1)
    contradiction_rate = contradicted_bundles / max(multi_q_bundles, 1)

    per_cell_accs = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro_acc = sum(per_cell_accs.values()) / max(len(per_cell_accs), 1)

    print(f"\n=== {split_name.upper()} FULL PAPER AUDIT ===")
    print(f"Overall Question Accuracy: {correct_q}/{total_q} = {overall_acc:.4f}")
    print(f"Unweighted Per-Cell Macro ({len(per_cell_accs)} cells): {macro_acc:.4f}")
    print(f"Joint Bundle Exact-Match Accuracy: {exact_bundles}/{total_bundles} = {bundle_exact_acc:.4f}")
    print(f"Multi-Question Contradiction Rate: {contradicted_bundles}/{multi_q_bundles} = {contradiction_rate:.4f}")

    return {
        "overall_acc": overall_acc,
        "macro_acc": macro_acc,
        "num_cells": len(per_cell_accs),
        "bundle_exact_acc": bundle_exact_acc,
        "contradiction_rate": contradiction_rate,
        "per_cell": per_cell_accs,
        "primitives": {k: c / max(t, 1) for k, (c, t) in prim_stats.items()}
    }

dev_audit = full_paper_audit(dev_bundles, "dev")
test_audit = full_paper_audit(test_bundles, "test")

witness['full_dev_audit'] = dev_audit
witness['full_test_audit'] = test_audit
with open(f"{OUT_DIR}/witness-v5-full.json", "w") as f:
    json.dump(witness, f, indent=2)
print("Saved complete paper witness to witness-v5-full.json!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned DeBERTa backbone and tokenizer to Google Drive!

=== DEV FULL PAPER AUDIT ===
Overall Question Accuracy: 1161/1440 = 0.8063
Unweighted Per-Cell Macro (14 cells): 0.8024
Joint Bundle Exact-Match Accuracy: 912/1176 = 0.7755
Multi-Question Contradiction Rate: 0/172 = 0.0000

=== TEST FULL PAPER AUDIT ===
Overall Question Accuracy: 1162/1440 = 0.8069
Unweighted Per-Cell Macro (14 cells): 0.7988
Joint Bundle Exact-Match Accuracy: 909/1176 = 0.7730
Multi-Question Contradiction Rate: 0/172 = 0.0000
Saved complete paper witness to witness-v5-full.json!


In [ ]:
# --- CELL 13: RESTORE v5 (0.8069) + TRUE 16-CELL AUDIT + STAGE 3 (v5.1) UPGRADE ---
import os, gc, copy, json, math, time, random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"

# 1. Load your ALREADY FINE-TUNED DeBERTa-v3-large backbone & tokenizer from Drive
print(f"1. Loading fine-tuned DeBERTa backbone from {OUT_DIR}/deberta-v5-finetuned ...")
tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
base = AutoModel.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned", torch_dtype=torch.float32).to(device)

MASK_ID = tok.mask_token_id
CLS_ID = tok.cls_token_id
SEP_ID = tok.sep_token_id
PAD_ID = tok.pad_token_id

def text_of(x):
    if isinstance(x, str): return x
    if x is None: return ''
    if isinstance(x, dict): return ' | '.join(f"{k}: {text_of(v)}" for k, v in x.items())
    if isinstance(x, list): return ' '.join(text_of(v) for v in x)
    return str(x)

def build_bundle_examples(bundle):
    state_text = text_of(bundle['state'])
    encoded_qs = []
    for q in bundle['questions']:
        n_opt = len(q['options'])
        max_opt_toks = max(2, (MAX_LEN - 96) // max(1, n_opt) - 2)
        opt_chunks = []
        for label, desc in zip(q['options'], q['descs']):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d and max_opt_toks > 4:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids'][:max_opt_toks]
            if not opt_ids: opt_ids = [tok.unk_token_id]
            opt_chunks.append(opt_ids + [MASK_ID, SEP_ID])

        prefix = f"Task ({bundle['source']} / {q['qname']}): {text_of(q['instructions'])} | Context: "
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids'][:64]
        state_ids = tok(state_text, add_special_tokens=False)['input_ids'][: (MAX_LEN - 64 - sum(len(c) for c in opt_chunks))]

        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID]
        mask_pos = []
        for chunk in opt_chunks:
            if len(ids) + len(chunk) > MAX_LEN: break
            ids += chunk
            mask_pos.append(len(ids) - 2)

        encoded_qs.append({
            'qname': q['qname'],
            'type': q['type'],
            'options': q['options'],
            'instructions': q['instructions'],
            'input_ids': ids,
            'mask_pos': mask_pos,
            'gold': int(q['gold']),
            'n_opt': len(mask_pos)
        })
    return {'source': bundle['source'], 'questions': encoded_qs}

def bundle_collate(bundle_list):
    max_q = max(len(b['questions']) for b in bundle_list)
    max_len = max(max(len(q['input_ids']) for q in b['questions']) for b in bundle_list)
    max_opt = max(max(len(q['mask_pos']) for q in b['questions']) for b in bundle_list)
    B = len(bundle_list)

    ids = torch.full((B, max_q, max_len), PAD_ID, dtype=torch.long)
    attn = torch.zeros((B, max_q, max_len), dtype=torch.long)
    mpos = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    omask = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    golds = []

    for i, b in enumerate(bundle_list):
        g_list = []
        for j, q in enumerate(b['questions']):
            ids[i, j, :len(q['input_ids'])] = torch.tensor(q['input_ids'])
            attn[i, j, :len(q['input_ids'])] = 1
            mpos[i, j, :len(q['mask_pos'])] = torch.tensor(q['mask_pos'])
            omask[i, j, :len(q['mask_pos'])] = 1
            g_list.append(int(q['gold']))
        golds.append(g_list)
    return ids, attn, mpos, omask, golds, bundle_list

# 2. Define QuestionGraphHead with Cross-Question Message Passing (zero-init gate preserves v5 at step 0)
class QuestionGraphHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )
        # Cross-Question Graph interaction layer over question-option states
        self.q_norm = nn.LayerNorm(hidden_size)
        self.q_attn = nn.MultiheadAttention(hidden_size, num_heads=8, dropout=0.1, batch_first=True)
        self.graph_gate = nn.Parameter(torch.zeros(1))  # Starts at 0.0 so Step 0 == exact v5 (0.8069)

    def forward(self, base_model, input_ids, attn, mask_pos, opt_mask, use_graph=True):
        B, M, L = input_ids.shape
        flat_ids = input_ids.view(B * M, L)
        flat_attn = attn.view(B * M, L)

        out = base_model(input_ids=flat_ids, attention_mask=flat_attn).last_hidden_state
        _, _, O = mask_pos.shape
        flat_mpos = mask_pos.view(B * M, O)

        idx = torch.arange(B * M, device=input_ids.device).unsqueeze(1)
        h_options = out[idx, flat_mpos].view(B, M, O, -1)  # (B, M, O, H)

        if use_graph and M > 1:
            # Cross-question message passing across all options in the document bundle
            flat_ho = h_options.view(B, M * O, -1)
            flat_mask = (opt_mask.view(B, M * O) == 0)
            norm_ho = self.q_norm(flat_ho)
            graph_out, _ = self.q_attn(norm_ho, norm_ho, norm_ho, key_padding_mask=flat_mask)
            h_options = h_options + torch.tanh(self.graph_gate) * graph_out.view(B, M, O, -1)

        logits = self.scorer(h_options).squeeze(-1).float()
        logits = logits.masked_fill(opt_mask.view(B, M, O) == 0, -1e9)
        return logits

model = QuestionGraphHead(base.config.hidden_size).to(device)
ckpt = torch.load(f"{OUT_DIR}/myna-v5-best.pt", map_location=device)
model.load_state_dict(ckpt["state"], strict=False)
print("2. Loaded myna-v5-best.pt into QuestionGraphHead!")

# 3. True 16-Cell Paper Audit (Fixes q['qname'] so all 16 cells of decision-v2-pilot are scored)
@torch.no_grad()
def full_paper_audit(bundles_raw, split_name="test", use_graph=True):
    model.eval()
    built = [build_bundle_examples(x) for x in bundles_raw]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)

    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q_bundles, contradicted_bundles = 0, 0
    cell_stats = defaultdict(lambda: [0, 0])
    prim_stats = defaultdict(lambda: [0, 0])
    score_mae_sum, score_count = 0.0, 0

    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask, use_graph=use_graph)

        for b_idx, b in enumerate(raw_b):
            qs = b['questions']
            b_all_correct = True
            preds_by_type = {}

            for j, q in enumerate(qs):
                n_opt = q['n_opt']
                pred = logits[b_idx, j, :n_opt].argmax(dim=-1).item()
                gold = int(golds[b_idx][j])
                ok = int(pred == gold)

                correct_q += ok
                total_q += 1
                if not ok:
                    b_all_correct = False

                ptype = q['type']
                prim_stats[ptype][0] += ok
                prim_stats[ptype][1] += 1

                if ptype == 'score':
                    score_mae_sum += abs(pred - gold) / max(n_opt - 1, 1)
                    score_count += 1

                # True (source, qname) 16-cell key matching myna.report
                src = b.get('source', 'unknown')
                qname = q['qname']
                cell_stats[(src, qname)][0] += ok
                cell_stats[(src, qname)][1] += 1
                preds_by_type[ptype] = (pred, q)

            total_bundles += 1
            if b_all_correct:
                exact_bundles += 1

            if len(qs) > 1:
                multi_q_bundles += 1
                if 'choice' in preds_by_type and 'noul' in preds_by_type:
                    c_pred, c_q = preds_by_type['choice']
                    n_pred, n_q = preds_by_type['noul']
                    c_opts = c_q.get('options', [])
                    if c_pred < len(c_opts):
                        chosen_label = str(c_opts[c_pred]).lower()
                        n_inst = str(n_q.get('instructions', '')).lower()
                        if chosen_label in n_inst and n_pred == 0:
                            contradicted_bundles += 1

    overall_acc = correct_q / max(total_q, 1)
    bundle_exact_acc = exact_bundles / max(total_bundles, 1)
    contradiction_rate = contradicted_bundles / max(multi_q_bundles, 1)
    per_cell_accs = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro_acc = sum(per_cell_accs.values()) / max(len(per_cell_accs), 1)
    prims = {k: c / max(t, 1) for k, (c, t) in prim_stats.items()}

    print(f"\n=== {split_name.upper()} TRUE 16-CELL PAPER AUDIT ===")
    print(f"Overall Accuracy: {correct_q}/{total_q} = {overall_acc:.4f} | {len(per_cell_accs)}-Cell Macro: {macro_acc:.4f}")
    print(f"Primitives -> choice: {prims.get('choice',0):.4f} | score: {prims.get('score',0):.4f} | noul: {prims.get('noul',0):.4f}")
    print(f"Joint Bundle Exact-Match: {exact_bundles}/{total_bundles} = {bundle_exact_acc:.4f} | Contradictions: {contradicted_bundles}/{multi_q_bundles} = {contradiction_rate:.4f}")
    return {
        "overall_acc": overall_acc, "macro_acc": macro_acc, "num_cells": len(per_cell_accs),
        "bundle_exact_acc": bundle_exact_acc, "contradiction_rate": contradiction_rate,
        "score_mae": score_mae_sum / max(score_count, 1),
        "per_cell": per_cell_accs, "primitives": prims
    }

print("3. Verifying restored v5 model on Dev and Test before Stage 3...")
v5_dev = full_paper_audit(dev_bundles, "v5_dev_baseline")
v5_test = full_paper_audit(test_bundles, "v5_test_baseline")

# 4. STAGE 3 (myna-v5.1): Ordinal EMD + Brier Calibration + Cross-Question Graph Polish
ref_model = copy.deepcopy(model).eval()
for p in ref_model.parameters():
    p.requires_grad = False

# Freeze bottom 18 layers of DeBERTa, fine-tune top 6 layers + QuestionGraphHead (fast & prevents overfitting)
for name, param in base.named_parameters():
    param.requires_grad = any(f"layer.{i}." in name for i in range(18, 24)) or ("pooler" in name)
if hasattr(base, "gradient_checkpointing_enable"):
    base.gradient_checkpointing_enable()

# Oversample 'score' and multi-question bundles so 'score' jumps from 0.6333 -> 0.72+
score_or_multi = [b for b in train_bundles if len(b['questions']) > 1 or any(q['type'] == 'score' for q in b['questions'])]
single_choice = [b for b in train_bundles if b not in score_or_multi]
random.seed(SEED)
stage3_raw = score_or_multi * 2 + random.sample(single_choice, min(len(single_choice), len(score_or_multi) * 2))
random.shuffle(stage3_raw)
stage3_built = [build_bundle_examples(x) for x in stage3_raw[:10000]]
print(f"\n4. Launching Stage 3 (v5.1) on {len(stage3_built)} balanced bundles (~18 mins on T4)...")

def stage3_loss_fn(logits, omask, golds, raw_bundles, ref_logits):
    losses = []
    for b_idx, b in enumerate(raw_bundles):
        qs = b['questions']
        probs_by_type = {}
        for j, q in enumerate(qs):
            n_opts = int(omask[b_idx, j].sum().item())
            if n_opts <= 1: continue
            q_logits = logits[b_idx, j, :n_opts].float()
            gold = int(golds[b_idx][j])
            if gold < 0 or gold >= n_opts: continue

            ce = F.cross_entropy(q_logits.unsqueeze(0), torch.tensor([gold], device=device))
            probs = F.softmax(q_logits, dim=-1)
            target_oh = F.one_hot(torch.tensor(gold, device=device), num_classes=n_opts).float()
            brier = torch.mean((probs - target_oh) ** 2)

            if q['type'] == 'score':
                # Ordinal Earth Mover's Distance (EMD) penalizes distance between predicted & gold levels
                cdf_p = torch.cumsum(probs, dim=-1)
                cdf_t = torch.cumsum(target_oh, dim=-1)
                emd = torch.mean((cdf_p - cdf_t) ** 2)
                q_loss = 0.55 * ce + 1.60 * emd + 0.25 * brier
            else:
                q_loss = 0.85 * ce + 0.25 * brier

            # KL leash to v5 so choice (0.7773) and noul (0.9306) stay protected
            ref_p = F.softmax(ref_logits[b_idx, j, :n_opts].float(), dim=-1).detach()
            kl = F.kl_div(F.log_softmax(q_logits, dim=-1), ref_p, reduction='batchmean')
            losses.append(q_loss + 0.15 * kl)
            probs_by_type[q['type']] = (probs, q)

        # Differentiable Cross-Question Consistency Loss on multi-question bundles
        if 'choice' in probs_by_type and 'noul' in probs_by_type:
            c_probs, c_q = probs_by_type['choice']
            n_probs, n_q = probs_by_type['noul']
            n_inst = str(n_q.get('instructions', '')).lower()
            for opt_i, opt_name in enumerate(c_q.get('options', [])):
                if opt_i < c_probs.size(0) and str(opt_name).lower() in n_inst:
                    # Penalize P(choice == X) * P(noul == False)
                    contradiction_prob = c_probs[opt_i] * n_probs[0]
                    losses.append(0.50 * contradiction_prob)

    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=device, requires_grad=True)

optimizer = torch.optim.AdamW([
    {'params': [p for p in base.parameters() if p.requires_grad], 'lr': 8e-6, 'weight_decay': 0.01},
    {'params': model.parameters(), 'lr': 6e-5, 'weight_decay': 0.01}
])
dl_s3 = DataLoader(stage3_built, batch_size=1, shuffle=True, collate_fn=bundle_collate)
total_steps = math.ceil(len(dl_s3) / 16)
sched = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))

best_dev_score = v5_dev["overall_acc"] + v5_dev["macro_acc"]
best_v51_saved = False
optimizer.zero_grad()
t0 = time.time()
running_loss = 0.0

for step, (ids, attn, mpos, omask, golds, raw_b) in enumerate(dl_s3, 1):
    model.train()
    base.train()
    ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
        with torch.no_grad():
            ref_logits = ref_model(base, ids, attn, mpos, omask, use_graph=False)
        logits = model(base, ids, attn, mpos, omask, use_graph=True)
        loss = stage3_loss_fn(logits, omask, golds, raw_b, ref_logits) / 16.0

    scaler.scale(loss).backward()
    running_loss += loss.item() * 16.0

    if step % 16 == 0 or step == len(dl_s3):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(base.parameters()) + list(model.parameters()), 1.0)
        scaler.step(optimizer)
        scaler.update()
        sched.step()
        optimizer.zero_grad()

    if step % 1000 == 0:
        print(f"Stage 3 step {step}/{len(dl_s3)} | loss: {running_loss/step:.4f} | elapsed: {time.time()-t0:.0f}s")

    if step % 2500 == 0 or step == len(dl_s3):
        dev_res = full_paper_audit(dev_bundles, f"v5.1_dev_step_{step}")
        cand_score = dev_res["overall_acc"] + dev_res["macro_acc"]
        if cand_score >= best_dev_score:
            best_dev_score = cand_score
            best_v51_saved = True
            torch.save({'state': model.state_dict(), 'dev_audit': dev_res}, f"{OUT_DIR}/myna-v5.1-best.pt")
            base.save_pretrained(f"{OUT_DIR}/deberta-v5.1-finetuned")
            tok.save_pretrained(f"{OUT_DIR}/deberta-v5.1-finetuned")
            print(f"  --> SAVED NEW BEST v5.1 CHECKPOINT (Overall: {dev_res['overall_acc']:.4f}, Macro: {dev_res['macro_acc']:.4f})")

if best_v51_saved:
    base = AutoModel.from_pretrained(f"{OUT_DIR}/deberta-v5.1-finetuned", torch_dtype=torch.float32).to(device)
    model.load_state_dict(torch.load(f"{OUT_DIR}/myna-v5.1-best.pt", map_location=device)['state'])
    v51_test = full_paper_audit(test_bundles, "v5.1_test_final")

1. Loading fine-tuned DeBERTa backbone from /content/drive/MyDrive/myna-v5-prototype/deberta-v5-finetuned ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

2. Loaded myna-v5-best.pt into QuestionGraphHead!
3. Verifying restored v5 model on Dev and Test before Stage 3...

=== V5_DEV_BASELINE TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1161/1440 = 0.8063 | 16-Cell Macro: 0.8160
Primitives -> choice: 0.7615 | score: 0.6542 | noul: 0.9405
Joint Bundle Exact-Match: 912/1176 = 0.7755 | Contradictions: 1/172 = 0.0058

=== V5_TEST_BASELINE TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1162/1440 = 0.8069 | 16-Cell Macro: 0.8173
Primitives -> choice: 0.7773 | score: 0.6333 | noul: 0.9306
Joint Bundle Exact-Match: 909/1176 = 0.7730 | Contradictions: 0/172 = 0.0000

4. Launching Stage 3 (v5.1) on 10000 balanced bundles (~18 mins on T4)...
Stage 3 step 1000/10000 | loss: 0.6188 | elapsed: 327s
Stage 3 step 2000/10000 | loss: 0.5790 | elapsed: 653s

=== V5.1_DEV_STEP_2500 TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1133/1440 = 0.7868 | 16-Cell Macro: 0.8001
Primitives -> choice: 0.7385 | score: 0.6292 | noul: 0.9286
Joint Bundle Exact-Match: 883

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- CELL 14: TEMPERATURE CALIBRATION (GATE G5) + REAL-LIFE USABLE ENGINE API ---
import numpy as np

@torch.no_grad()
def collect_logits_and_golds(bundles_raw):
    model.eval()
    built = [build_bundle_examples(x) for x in bundles_raw]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    records = []
    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask, use_graph=True)
        for b_idx, b in enumerate(raw_b):
            for j, q in enumerate(b['questions']):
                n_opt = q['n_opt']
                l_vec = logits[b_idx, j, :n_opt].float().cpu()
                records.append({
                    'source': b['source'], 'qname': q['qname'], 'type': q['type'],
                    'logits': l_vec, 'gold': int(golds[b_idx][j])
                })
    return records

dev_recs = collect_logits_and_golds(dev_bundles)
test_recs = collect_logits_and_golds(test_bundles)

# 1. Fit optimal temperature per primitive type on Dev to minimize NLL / ECE
temps = {}
for ptype in ['choice', 'score', 'noul']:
    sub = [r for r in dev_recs if r['type'] == ptype]
    best_t, best_nll = 1.0, 1e9
    for t in np.linspace(0.5, 3.0, 51):
        nll = np.mean([F.cross_entropy((r['logits'] / t).unsqueeze(0), torch.tensor([r['gold']])).item() for r in sub])
        if nll < best_nll:
            best_nll, best_t = nll, float(t)
    temps[ptype] = round(best_t, 3)
print("Fitted Per-Primitive Temperatures on Dev:", temps)

# 2. Compute Calibrated Test ECE, Brier Score, and Gate G5 Risk-Coverage Curve
def evaluate_calibration(recs, temps_map):
    confs, corrects, briers = [], [], []
    g5_confs, g5_corrects = [], []
    for r in recs:
        t = temps_map.get(r['type'], 1.0)
        probs = F.softmax(r['logits'] / t, dim=-1).numpy()
        pred = int(np.argmax(probs))
        conf = float(np.max(probs))
        ok = int(pred == r['gold'])
        oh = np.zeros_like(probs)
        oh[r['gold']] = 1.0
        briers.append(float(np.mean((probs - oh) ** 2)))
        confs.append(conf)
        corrects.append(ok)
        if r['source'] in ('banking77', 'dbpedia14', 'trec'):
            g5_confs.append(conf)
            g5_corrects.append(ok)

    confs, corrects = np.array(confs), np.array(corrects)
    ece = 0.0
    for b in range(10):
        lo, hi = b / 10.0, (b + 1) / 10.0
        m = (confs >= lo) & (confs < hi if b < 9 else confs <= hi)
        if np.any(m):
            ece += (np.sum(m) / len(confs)) * abs(np.mean(corrects[m]) - np.mean(confs[m]))

    # Gate G5 check: accuracy at >= 60% coverage on banking77 + dbpedia14 + trec
    g5_order = np.argsort(-np.array(g5_confs))
    g5_top60 = np.array(g5_corrects)[g5_order[:int(0.60 * len(g5_order))]]
    g5_acc_60 = float(np.mean(g5_top60))
    print(f"Calibrated Test ECE: {ece:.4f} | Mean Brier: {np.mean(briers):.4f} | Gate G5 Acc @ 60% cov: {g5_acc_60:.4f}")
    return {"temperatures": temps_map, "test_ece": float(ece), "test_brier": float(np.mean(briers)), "g5_acc_at_60pct_cov": g5_acc_60}

cal_report = evaluate_calibration(test_recs, temps)

# Save full paper witness
with open(f"{OUT_DIR}/witness-v5.1-paper.json", "w") as f:
    json.dump({"temperatures": temps, "calibration": cal_report, "test_16cell": full_paper_audit(test_bundles, "final_paper")}, f, indent=2)
print(f"Saved complete paper witness to {OUT_DIR}/witness-v5.1-paper.json!")

# 3. REAL-LIFE USABLE INTERFACE (matches your GitHub README: predict / observe / append / ask)
class MynaSession:
    def __init__(self, engine, state_text):
        self.engine = engine
        self.state_text = state_text

    def append(self, delta_text):
        self.state_text = (self.state_text + "\n" + text_of(delta_text)).strip()
        return self

    def ask(self, questions_dict, abstain_below=0.55):
        return self.engine.predict(self.state_text, questions_dict, abstain_below=abstain_below)

class MynaV5Engine:
    def __init__(self, base_model, head_model, tokenizer, temperatures):
        self.base = base_model.eval()
        self.head = head_model.eval()
        self.tok = tokenizer
        self.temps = temperatures

    def observe(self, state):
        return MynaSession(self, text_of(state))

    @torch.no_grad()
    def predict(self, state, questions_dict, abstain_below=0.55):
        q_list = []
        for qname, q in questions_dict.items():
            qtype = q['type']
            if qtype == 'choice':
                crit = q['criteria']
                opts = list(crit.keys()) if isinstance(crit, dict) else list(crit)
                descs = [crit[o] for o in opts] if isinstance(crit, dict) else ['' for _ in opts]
            elif qtype == 'noul':
                opts, descs = ['false', 'true'], ['statement is false', 'statement is true']
            elif qtype == 'score':
                opts = [str(i) for i in range(len(q['criteria']))]
                descs = list(q['criteria'])
            q_list.append({'qname': qname, 'type': qtype, 'instructions': q.get('instructions', ''), 'options': opts, 'descs': descs, 'gold': 0})

        built = build_bundle_examples({'source': 'live', 'state': state, 'questions': q_list})
        ids, attn, mpos, omask, _, _ = bundle_collate([built])
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = self.head(self.base, ids, attn, mpos, omask, use_graph=True)

        answers = {}
        for j, q in enumerate(q_list):
            n_opt = len(q['options'])
            t = self.temps.get(q['type'], 1.0)
            probs = F.softmax(logits[0, j, :n_opt].float() / t, dim=-1).cpu().numpy()
            top_idx = int(np.argmax(probs))
            conf = float(probs[top_idx])
            ans = {'confidence': round(conf, 4), 'abstain': bool(conf < abstain_below)}
            if q['type'] == 'choice':
                ans['choice'] = q['options'][top_idx]
                ans['probs'] = {q['options'][i]: round(float(probs[i]), 4) for i in range(n_opt)}
            elif q['type'] == 'noul':
                ans['noul'] = round(float(probs[1]), 4)
                ans['value'] = bool(top_idx == 1)
            elif q['type'] == 'score':
                expected_score = float(sum(i * probs[i] for i in range(n_opt)))
                ans['level'] = top_idx
                ans['label'] = q['descs'][top_idx]
                ans['score'] = round(expected_score, 4)
            answers[q['qname']] = ans
        return {'answers': answers}

# Smoke-test your real-life engine on the exact example from your GitHub README!
myna_live = MynaV5Engine(base, model, tok, temps)
obs = myna_live.observe("Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.")
obs.append("And the second charge is still showing on my card this morning.")
live_out = obs.ask({
    "department": {"type": "choice", "instructions": "Which department should handle this?",
                   "criteria": {"billing": "invoices, payments, refunds", "technical": "bugs, outages, system errors"}},
    "urgency": {"type": "score", "instructions": "How urgent is this?",
                "criteria": ["not urgent", "soon", "blocking", "drop everything"]},
    "churn_risk": {"type": "noul", "instructions": "Does the writer threaten to cancel?"}
})
print("\n=== REAL-LIFE ENGINE LIVE TEST ===")
print(json.dumps(live_out, indent=2))

Fitted Per-Primitive Temperatures on Dev: {'choice': 1.55, 'score': 1.4, 'noul': 1.6}
Calibrated Test ECE: 0.0268 | Mean Brier: 0.0475 | Gate G5 Acc @ 60% cov: 0.9856

=== FINAL_PAPER TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1152/1440 = 0.8000 | 16-Cell Macro: 0.8109
Primitives -> choice: 0.7716 | score: 0.6208 | noul: 0.9246
Joint Bundle Exact-Match: 900/1176 = 0.7653 | Contradictions: 0/172 = 0.0000
Saved complete paper witness to /content/drive/MyDrive/myna-v5-prototype/witness-v5.1-paper.json!

=== REAL-LIFE ENGINE LIVE TEST ===
{
  "answers": {
    "department": {
      "confidence": 0.8671,
      "abstain": false,
      "choice": "billing",
      "probs": {
        "billing": 0.8671,
        "technical": 0.1329
      }
    },
    "urgency": {
      "confidence": 0.4021,
      "abstain": true,
      "level": 1,
      "label": "soon",
      "score": 1.6079
    },
    "churn_risk": {
      "confidence": 0.9347,
      "abstain": false,
      "noul": 0.9347,
      "value": true


In [ ]:
# --- CELL 15: LOCK IN PRISTINE 0.8173 MACRO CHECKPOINT + CALIBRATION + 16-CELL TABLE FOR PAPER ---
import json
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"

# 1. Reload pristine v5 (1162/1440 = 0.8069 overall, 0.8173 16-cell macro) from Google Drive
print("Reloading pristine v5 checkpoint from Google Drive...")
tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
base = AutoModel.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned", dtype=torch.float32).to(device).eval()

ckpt = torch.load(f"{OUT_DIR}/myna-v5-best.pt", map_location=device)
model.load_state_dict(ckpt["state"], strict=False)
model.to(device).eval()

# 2. Run True 16-Cell Audit on Dev and Test
dev_16 = full_paper_audit(dev_bundles, "v5_pristine_dev", use_graph=False)
test_16 = full_paper_audit(test_bundles, "v5_pristine_test", use_graph=False)

# 3. Fit Per-Primitive Temperatures on Pristine v5 Dev & Score Gate G5 on Test
dev_recs = collect_logits_and_golds(dev_bundles)
test_recs = collect_logits_and_golds(test_bundles)

temps = {}
for ptype in ['choice', 'score', 'noul']:
    sub = [r for r in dev_recs if r['type'] == ptype]
    best_t, best_nll = 1.0, 1e9
    for t in np.linspace(0.5, 3.0, 51):
        nll = np.mean([F.cross_entropy((r['logits'] / t).unsqueeze(0), torch.tensor([r['gold']])).item() for r in sub])
        if nll < best_nll:
            best_nll, best_t = nll, float(t)
    temps[ptype] = round(best_t, 3)

cal_report = evaluate_calibration(test_recs, temps)

print("\n=== TABLE 1: ALL 16 CELLS ON DECISION-V2 TEST ===")
for cell_key, acc in sorted(test_16["per_cell"].items()):
    print(f"  {cell_key:35s} : {acc:.4f}")

# 4. Save Master Paper Witness JSON to Drive
master_witness = {
    "model_label": "myna-v5 (DeBERTa-v3-large + Multi-Primitive Question-Graph Head)",
    "base_model": "microsoft/deberta-v3-large",
    "seed": 42,
    "temperatures": temps,
    "calibration_and_g5": cal_report,
    "dev_16cell_audit": dev_16,
    "test_16cell_audit": test_16
}
with open(f"{OUT_DIR}/witness-v5-master-paper.json", "w") as f:
    json.dump(master_witness, f, indent=2)
print(f"\nSaved master paper witness to {OUT_DIR}/witness-v5-master-paper.json!")

Reloading pristine v5 checkpoint from Google Drive...


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]


=== V5_PRISTINE_DEV TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1161/1440 = 0.8063 | 16-Cell Macro: 0.8160
Primitives -> choice: 0.7615 | score: 0.6542 | noul: 0.9405
Joint Bundle Exact-Match: 912/1176 = 0.7755 | Contradictions: 1/172 = 0.0058

=== V5_PRISTINE_TEST TRUE 16-CELL PAPER AUDIT ===
Overall Accuracy: 1162/1440 = 0.8069 | 16-Cell Macro: 0.8173
Primitives -> choice: 0.7773 | score: 0.6333 | noul: 0.9306
Joint Bundle Exact-Match: 909/1176 = 0.7730 | Contradictions: 0/172 = 0.0000
Calibrated Test ECE: 0.0204 | Mean Brier: 0.0445 | Gate G5 Acc @ 60% cov: 0.9808

=== TABLE 1: ALL 16 CELLS ON DECISION-V2 TEST ===
  agnews/is_business                  : 0.9535
  agnews/is_scitech                   : 0.9184
  agnews/is_sports                    : 0.9787
  agnews/is_world                     : 0.9778
  agnews/topic                        : 0.9052
  amazon/stars                        : 0.6375
  banking77/intent                    : 0.0086
  boolq/answer                        : 0.

In [ ]:
# --- NEW CELL: RESTORE PRISTINE v5 FROM DRIVE (no training) ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"

tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
base = AutoModel.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned", torch_dtype=torch.float32).to(device)

MASK_ID = tok.mask_token_id
CLS_ID = tok.cls_token_id
SEP_ID = tok.sep_token_id
PAD_ID = tok.pad_token_id

def text_of(x):
    if isinstance(x, str): return x
    if x is None: return ''
    if isinstance(x, dict): return ' | '.join(f"{k}: {text_of(v)}" for k, v in x.items())
    if isinstance(x, list): return ' '.join(text_of(v) for v in x)
    return str(x)

def build_bundle_examples(bundle):
    state_text = text_of(bundle['state'])
    encoded_qs = []
    for q in bundle['questions']:
        n_opt = len(q['options'])
        max_opt_toks = max(2, (MAX_LEN - 96) // max(1, n_opt) - 2)
        opt_chunks = []
        for label, desc in zip(q['options'], q['descs']):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d and max_opt_toks > 4:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids'][:max_opt_toks]
            if not opt_ids: opt_ids = [tok.unk_token_id]
            opt_chunks.append(opt_ids + [MASK_ID, SEP_ID])
        prefix = f"Task ({bundle['source']} / {q['qname']}): {text_of(q['instructions'])} | Context: "
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids'][:64]
        state_ids = tok(state_text, add_special_tokens=False)['input_ids'][: (MAX_LEN - 64 - sum(len(c) for c in opt_chunks))]
        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID]
        mask_pos = []
        for chunk in opt_chunks:
            if len(ids) + len(chunk) > MAX_LEN: break
            ids += chunk
            mask_pos.append(len(ids) - 2)
        encoded_qs.append({
            'qname': q['qname'], 'type': q['type'], 'options': q['options'],
            'instructions': q['instructions'], 'input_ids': ids,
            'mask_pos': mask_pos, 'gold': int(q['gold']), 'n_opt': len(mask_pos)
        })
    return {'source': bundle['source'], 'questions': encoded_qs}

def bundle_collate(bundle_list):
    max_q = max(len(b['questions']) for b in bundle_list)
    max_len = max(max(len(q['input_ids']) for q in b['questions']) for b in bundle_list)
    max_opt = max(max(len(q['mask_pos']) for q in b['questions']) for b in bundle_list)
    B = len(bundle_list)
    ids = torch.full((B, max_q, max_len), PAD_ID, dtype=torch.long)
    attn = torch.zeros((B, max_q, max_len), dtype=torch.long)
    mpos = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    omask = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    golds = []
    for i, b in enumerate(bundle_list):
        g_list = []
        for j, q in enumerate(b['questions']):
            ids[i, j, :len(q['input_ids'])] = torch.tensor(q['input_ids'])
            attn[i, j, :len(q['input_ids'])] = 1
            mpos[i, j, :len(q['mask_pos'])] = torch.tensor(q['mask_pos'])
            omask[i, j, :len(q['mask_pos'])] = 1
            g_list.append(int(q['gold']))
        golds.append(g_list)
    return ids, attn, mpos, omask, golds, bundle_list

class QuestionGraphHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )
        self.q_norm = nn.LayerNorm(hidden_size)
        self.q_attn = nn.MultiheadAttention(hidden_size, num_heads=8, dropout=0.1, batch_first=True)
        self.graph_gate = nn.Parameter(torch.zeros(1))

    def forward(self, base_model, input_ids, attn, mask_pos, opt_mask, use_graph=True):
        B, M, L = input_ids.shape
        flat_ids = input_ids.view(B * M, L)
        flat_attn = attn.view(B * M, L)
        out = base_model(input_ids=flat_ids, attention_mask=flat_attn).last_hidden_state
        _, _, O = mask_pos.shape
        flat_mpos = mask_pos.view(B * M, O)
        idx = torch.arange(B * M, device=input_ids.device).unsqueeze(1)
        h_options = out[idx, flat_mpos].view(B, M, O, -1)
        if use_graph and M > 1:
            flat_ho = h_options.view(B, M * O, -1)
            flat_mask = (opt_mask.view(B, M * O) == 0)
            norm_ho = self.q_norm(flat_ho)
            graph_out, _ = self.q_attn(norm_ho, norm_ho, norm_ho, key_padding_mask=flat_mask)
            h_options = h_options + torch.tanh(self.graph_gate) * graph_out.view(B, M, O, -1)
        logits = self.scorer(h_options).squeeze(-1).float()
        logits = logits.masked_fill(opt_mask.view(B, M, O) == 0, -1e9)
        return logits

model = QuestionGraphHead(base.config.hidden_size).to(device)
ckpt = torch.load(f"{OUT_DIR}/myna-v5-best.pt", map_location=device)
model.load_state_dict(ckpt["state"], strict=False)
model.eval()
base.eval()
print("Restored pristine v5 from Drive. Ready for diagnostic.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Restored pristine v5 from Drive. Ready for diagnostic.


In [ ]:
# --- CORRECTED BANKING77 DIAGNOSTIC ---
import collections, json

bank_raw = [b for b in test_bundles if b.get('source') == 'banking77']
print("banking77 test bundles:", len(bank_raw))

variants = []
with open('data/decision-v2-pilot/test.jsonl', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line)
        if d['_meta']['source'] == 'banking77':
            variants.append(d['_meta'].get('variant', 'unknown'))
print("variants:", collections.Counter(variants))

built = [build_bundle_examples(x) for x in bank_raw]
dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)

rows = []
with torch.no_grad():
    model.eval()
    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask, use_graph=True)
        for b_idx, b in enumerate(raw_b):
            for j, q in enumerate(b['questions']):
                if q.get('type') != 'choice':
                    continue
                opts = q.get('options', [])
                n_opt = int(q.get('n_opt', len(opts)))
                gold = int(golds[b_idx][j])
                pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                rows.append((gold, pred,
                             opts[gold] if gold < len(opts) else f"?idx{gold}",
                             opts[pred] if pred < len(opts) else f"?idx{pred}",
                             n_opt, len(opts), 'unknown'))

for i, r in enumerate(rows):
    if i < len(variants):
        rows[i] = r[:6] + (variants[i],)

print(f"banking77/intent: {sum(1 for g, p, *_ in rows if g == p)}/{len(rows)}")
print("n_opt distribution:", collections.Counter(r[4] for r in rows))
print("bundles where n_opt < n_options:", sum(1 for r in rows if r[4] < r[5]))
print("predicted labels (top 10):", collections.Counter(r[3] for r in rows).most_common(10))
for lo, hi in [(0, 20), (20, 40), (40, 60), (60, 80)]:
    sub = [r for r in rows if lo <= r[0] < hi]
    if sub:
        print(f"gold index {lo}-{hi-1}: {sum(1 for g, p, *_ in sub if g == p)}/{len(sub)} correct")
for var in sorted(set(r[6] for r in rows)):
    sub = [r for r in rows if r[6] == var]
    print(f"variant {var}: {sum(1 for g, p, *_ in sub if g == p)}/{len(sub)} correct")
print("correct rows:")
for r in rows:
    if r[0] == r[1]:
        print(" ", r)


banking77 test bundles: 116
variants: Counter({'clean': 80, 'none_present': 12, 'none_absent': 12, 'permuted': 12})
banking77/intent: 1/116
n_opt distribution: Counter({77: 104, 78: 12})
bundles where n_opt < n_options: 0
predicted labels (top 10): [('card_payment_wrong_exchange_rate', 6), ('apple_pay_or_google_pay', 6), ('compromised_card', 6), ('age_limit', 6), ('automatic_top_up', 4), ('card_linking', 4), ('pending_card_payment', 4), ('card_not_working', 4), ('card_about_to_expire', 3), ('getting_virtual_card', 3)]
gold index 0-19: 1/28 correct
gold index 20-39: 0/35 correct
gold index 40-59: 0/31 correct
gold index 60-79: 0/22 correct
variant clean: 1/80 correct
variant none_absent: 0/12 correct
variant none_present: 0/12 correct
variant permuted: 0/12 correct
correct rows:
  (4, 4, 'automatic_top_up', 'automatic_top_up', 77, 77, 'clean')


In [ ]:
# --- BANKING77 TRAIN CHECK (same model, no retraining) ---
import collections

bank_train = [b for b in train_bundles if b.get('source') == 'banking77'][:200]
print("banking77 train bundles sampled:", len(bank_train))

built = [build_bundle_examples(x) for x in bank_train]
dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)

rows = []
with torch.no_grad():
    model.eval()
    for ids, attn, mpos, omask, golds, raw_b in dl:
        ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            logits = model(base, ids, attn, mpos, omask, use_graph=True)
        for b_idx, b in enumerate(raw_b):
            for j, q in enumerate(b['questions']):
                if q.get('type') != 'choice':
                    continue
                opts = q.get('options', [])
                n_opt = int(q.get('n_opt', len(opts)))
                gold = int(golds[b_idx][j])
                pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                rows.append((gold, pred))

print(f"banking77 train: {sum(1 for g, p in rows if g == p)}/{len(rows)}")
print("predicted labels (top 10):", collections.Counter(p for _, p in rows).most_common(10))


banking77 train bundles sampled: 200
banking77 train: 2/200
predicted labels (top 10): [(0, 11), (2, 10), (4, 9), (3, 9), (1, 9), (19, 9), (11, 8), (7, 8), (6, 7), (5, 7)]


In [ ]:
# --- PER-OPTION PROBE: banking77, one option at a time, existing scorer, no training ---
import collections, torch

bank_test = [b for b in test_bundles if b.get('source') == 'banking77']
print("banking77 test bundles:", len(bank_test))

items = []
for bi, bundle in enumerate(bank_test):
    q = bundle['questions'][0]
    prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
    state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
    for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
        opt_text = label.replace('_', ' ')
        d = text_of(desc)
        if d:
            opt_text += ': ' + ' '.join(d.split()[:10])
        opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
        if not opt_ids:
            opt_ids = [tok.unk_token_id]
        state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
        items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2, 'gold': int(q['gold']), 'label': label})

print("total option encodings:", len(items))

scores = [0.0] * len(items)
model.eval()
base.eval()
with torch.no_grad():
    for start in range(0, len(items), 32):
        batch = items[start:start + 32]
        maxlen = max(len(x['ids']) for x in batch)
        ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
        attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
        mpos_t = torch.zeros(len(batch), dtype=torch.long)
        for i, x in enumerate(batch):
            ids_t[i, :len(x['ids'])] = torch.tensor(x['ids'])
            attn_t[i, :len(x['ids'])] = 1
            mpos_t[i] = x['mask']
        ids_t, attn_t = ids_t.to(device), attn_t.to(device)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
            h = out[torch.arange(len(batch), device=device), mpos_t.to(device)]
            s = model.scorer(h).squeeze(-1).float()
        for i, x in enumerate(batch):
            scores[start + i] = float(s[i].item())
        if (start // 32 + 1) % 50 == 0:
            print(f"scored {start + len(batch)}/{len(items)} options")

correct = 0
pred_labels = []
idx = 0
for bundle in bank_test:
    q = bundle['questions'][0]
    n = len(q['options'])
    bundle_scores = scores[idx:idx + n]
    idx += n
    pred = int(max(range(n), key=lambda k: bundle_scores[k]))
    gold = int(q['gold'])
    if pred == gold:
        correct += 1
    pred_labels.append(q['options'][pred])

print(f"\nPER-OPTION PROBE banking77 test: {correct}/{len(bank_test)} = {correct / len(bank_test):.4f}")
print("predicted labels (top 10):", collections.Counter(pred_labels).most_common(10))


banking77 test bundles: 116
total option encodings: 8944
scored 1600/8944 options
scored 3200/8944 options
scored 4800/8944 options
scored 6400/8944 options
scored 8000/8944 options

PER-OPTION PROBE banking77 test: 25/116 = 0.2155
predicted labels (top 10): [('request_refund', 12), ('get_disposable_virtual_card', 7), ('virtual_card_not_working', 7), ('declined_card_payment', 5), ('compromised_card', 5), ('card_delivery_estimate', 5), ('verify_top_up', 4), ('declined_transfer', 4), ('apple_pay_or_google_pay', 4), ('card_not_working', 4)]


In [ ]:
# --- TRAIN A PER-OPTION SCORER FOR BANKING77 (frozen backbone, cached hiddens) ---
import collections, copy, random, time, torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(42)
bank_train_all = [b for b in train_bundles if b.get('source') == 'banking77']
bank_dev = [b for b in dev_bundles if b.get('source') == 'banking77']
bank_test = [b for b in test_bundles if b.get('source') == 'banking77']
train_sample = random.sample(bank_train_all, 1000)
print(f"train sample: {len(train_sample)} | dev: {len(bank_dev)} | test: {len(bank_test)}")

def build_per_option_items(bundles):
    items = []
    for bi, bundle in enumerate(bundles):
        q = bundle['questions'][0]
        prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
        state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
        for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
            if not opt_ids:
                opt_ids = [tok.unk_token_id]
            state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
            ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
            items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2})
    return items

def precompute_hiddens(bundles, split_name):
    items = build_per_option_items(bundles)
    print(f"{split_name}: {len(items)} option encodings")
    bundle_hiddens = []
    bundle_golds = []
    for bundle in bundles:
        q = bundle['questions'][0]
        bundle_golds.append(int(q['gold']))
        bundle_hiddens.append(torch.zeros(len(q['options']), base.config.hidden_size))
    base.eval()
    with torch.no_grad():
        for start in range(0, len(items), 32):
            batch = items[start:start + 32]
            maxlen = max(len(x['ids']) for x in batch)
            ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
            attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
            mpos_t = torch.zeros(len(batch), dtype=torch.long)
            for i, x in enumerate(batch):
                ids_t[i, :len(x['ids'])] = torch.tensor(x['ids'])
                attn_t[i, :len(x['ids'])] = 1
                mpos_t[i] = x['mask']
            ids_t, attn_t = ids_t.to(device), attn_t.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
                h = out[torch.arange(len(batch), device=device), mpos_t.to(device)].float().cpu()
            for i, x in enumerate(batch):
                bundle_hiddens[x['bundle']][x['opt']] = h[i]
            if (start // 32 + 1) % 50 == 0:
                print(f"  {split_name} {start + len(batch)}/{len(items)}")
    return bundle_hiddens, bundle_golds

t0 = time.time()
train_h, train_g = precompute_hiddens(train_sample, "train")
dev_h, dev_g = precompute_hiddens(bank_dev, "dev")
test_h, test_g = precompute_hiddens(bank_test, "test")
print(f"precompute done in {time.time() - t0:.0f}s")

def eval_cached(hiddens, golds, scorer):
    scorer.eval()
    correct = 0
    with torch.no_grad():
        for h, gold in zip(hiddens, golds):
            s = scorer(h.to(device)).squeeze(-1)
            if int(s.argmax().item()) == gold:
                correct += 1
    return correct / len(golds)

scorer = copy.deepcopy(model.scorer).to(device)
opt = torch.optim.AdamW(scorer.parameters(), lr=5e-4, weight_decay=0.01)
best_dev = 0.0
best_state = None
for epoch in range(5):
    scorer.train()
    order = list(range(len(train_h)))
    random.shuffle(order)
    tot_loss, n = 0.0, 0
    for start in range(0, len(order), 32):
        idx = order[start:start + 32]
        # bundles in train sample all have 77 options, so stack is safe
        h_batch = torch.stack([train_h[i] for i in idx]).to(device)
        g_batch = torch.tensor([train_g[i] for i in idx], dtype=torch.long, device=device)
        scores = scorer(h_batch).squeeze(-1)
        loss = F.cross_entropy(scores, g_batch)
        opt.zero_grad()
        loss.backward()
        opt.step()
        tot_loss += float(loss.item())
        n += 1
    dev_acc = eval_cached(dev_h, dev_g, scorer)
    print(f"epoch {epoch + 1}/5 loss {tot_loss / max(n, 1):.4f} | dev banking77: {dev_acc:.4f}")
    if dev_acc > best_dev:
        best_dev = dev_acc
        best_state = copy.deepcopy(scorer.state_dict())

scorer.load_state_dict(best_state)
test_acc = eval_cached(test_h, test_g, scorer)
print(f"\nBEST dev banking77: {best_dev:.4f} | FINAL test banking77 (per-option trained scorer): {test_acc:.4f}")
torch.save({'state': best_state, 'dev_acc': best_dev, 'test_acc': test_acc}, f"{OUT_DIR}/banking77-peroption-scorer.pt")
print("Saved separate scorer to banking77-peroption-scorer.pt (pristine model untouched)")


train sample: 1000 | dev: 116 | test: 116
train: 77000 option encodings
  train 1600/77000
  train 3200/77000
  train 4800/77000
  train 6400/77000
  train 8000/77000
  train 9600/77000
  train 11200/77000
  train 12800/77000
  train 14400/77000
  train 16000/77000
  train 17600/77000
  train 19200/77000
  train 20800/77000
  train 22400/77000
  train 24000/77000
  train 25600/77000
  train 27200/77000
  train 28800/77000
  train 30400/77000
  train 32000/77000
  train 33600/77000
  train 35200/77000
  train 36800/77000
  train 38400/77000
  train 40000/77000
  train 41600/77000
  train 43200/77000
  train 44800/77000
  train 46400/77000
  train 48000/77000
  train 49600/77000
  train 51200/77000
  train 52800/77000
  train 54400/77000
  train 56000/77000
  train 57600/77000
  train 59200/77000
  train 60800/77000
  train 62400/77000
  train 64000/77000
  train 65600/77000
  train 67200/77000
  train 68800/77000
  train 70400/77000
  train 72000/77000
  train 73600/77000
  train 75200/

In [ ]:
# --- JOINT HYBRID AUDIT: pristine packed + per-option banking77, dev + test ---
import collections, copy, json, torch
import torch.nn.functional as F

# reload the trained per-option scorer (separate file, pristine model untouched)
peroption_scorer = copy.deepcopy(model.scorer).to(device)
_hybrid_ckpt = torch.load(f"{OUT_DIR}/banking77-peroption-scorer.pt", map_location=device)
_hybrid_state = _hybrid_ckpt['state'] if isinstance(_hybrid_ckpt, dict) and 'state' in _hybrid_ckpt else _hybrid_ckpt
peroption_scorer.load_state_dict(_hybrid_state)
peroption_scorer.eval()
model.eval()
base.eval()
print("Loaded per-option scorer from banking77-peroption-scorer.pt")

def load_variants(path):
    variants = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['_meta']['source'] == 'banking77':
                variants.append(d['_meta'].get('variant', 'unknown'))
    return variants

def hybrid_audit(bundles_raw, variant_path, split_name):
    variants = load_variants(variant_path)
    bank_bundles = [b for b in bundles_raw if b.get('source') == 'banking77']
    other_bundles = [b for b in bundles_raw if b.get('source') != 'banking77']
    print(f"{split_name}: other {len(other_bundles)} bundles, banking77 {len(bank_bundles)} bundles, variants {collections.Counter(variants)}")

    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q_bundles, contradicted_bundles = 0, 0
    cell_stats = collections.defaultdict(lambda: [0, 0])
    prim_stats = collections.defaultdict(lambda: [0, 0])
    bank_variant_stats = collections.defaultdict(lambda: [0, 0])

    # Phase A: pristine packed model for all non-banking77 bundles
    built = [build_bundle_examples(x) for x in other_bundles]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    with torch.no_grad():
        for ids, attn, mpos, omask, golds, raw_b in dl:
            ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(base, ids, attn, mpos, omask, use_graph=True)
            for b_idx, b in enumerate(raw_b):
                qs = b['questions']
                b_all_correct = True
                preds_by_type = {}
                for j, q in enumerate(qs):
                    n_opt = int(q['n_opt'])
                    pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                    gold = int(golds[b_idx][j])
                    ok = int(pred == gold)
                    correct_q += ok
                    total_q += 1
                    if not ok:
                        b_all_correct = False
                    prim_stats[q['type']][0] += ok
                    prim_stats[q['type']][1] += 1
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][0] += ok
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][1] += 1
                    preds_by_type[q['type']] = (pred, q)
                total_bundles += 1
                if b_all_correct:
                    exact_bundles += 1
                if len(qs) > 1:
                    multi_q_bundles += 1
                    if 'choice' in preds_by_type and 'noul' in preds_by_type:
                        c_pred, c_q = preds_by_type['choice']
                        n_pred, n_q = preds_by_type['noul']
                        c_opts = c_q.get('options', [])
                        if c_pred < len(c_opts):
                            chosen = str(c_opts[c_pred]).lower()
                            n_inst = str(n_q.get('instructions', '')).lower()
                            if chosen in n_inst and n_pred == 0:
                                contradicted_bundles += 1

    # Phase B: per-option trained scorer for banking77 bundles
    bank_items = []
    for bi, bundle in enumerate(bank_bundles):
        q = bundle['questions'][0]
        prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
        state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
        for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
            if not opt_ids:
                opt_ids = [tok.unk_token_id]
            state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
            ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
            bank_items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2})

    bank_scores = [0.0] * len(bank_items)
    with torch.no_grad():
        for start in range(0, len(bank_items), 32):
            batch = bank_items[start:start + 32]
            maxlen = max(len(x['ids']) for x in batch)
            ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
            attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
            mpos_t = torch.zeros(len(batch), dtype=torch.long)
            for i, x in enumerate(batch):
                ids_t[i, :len(x['ids'])] = torch.tensor(x['ids'])
                attn_t[i, :len(x['ids'])] = 1
                mpos_t[i] = x['mask']
            ids_t, attn_t = ids_t.to(device), attn_t.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
                h = out[torch.arange(len(batch), device=device), mpos_t.to(device)]
                s = peroption_scorer(h).squeeze(-1).float()
            for i, x in enumerate(batch):
                bank_scores[start + i] = float(s[i].item())

    bank_idx = 0
    for bi, bundle in enumerate(bank_bundles):
        q = bundle['questions'][0]
        n = len(q['options'])
        bundle_scores = bank_scores[bank_idx:bank_idx + n]
        bank_idx += n
        pred = int(max(range(n), key=lambda k: bundle_scores[k]))
        gold = int(q['gold'])
        ok = int(pred == gold)
        correct_q += ok
        total_q += 1
        total_bundles += 1
        if ok:
            exact_bundles += 1
        prim_stats['choice'][0] += ok
        prim_stats['choice'][1] += 1
        cell_stats[('banking77', 'intent')][0] += ok
        cell_stats[('banking77', 'intent')][1] += 1
        var = variants[bi] if bi < len(variants) else 'unknown'
        bank_variant_stats[var][0] += ok
        bank_variant_stats[var][1] += 1

    overall = correct_q / max(total_q, 1)
    per_cell = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro = sum(per_cell.values()) / max(len(per_cell), 1)
    print(f"\n=== HYBRID {split_name.upper()} ===")
    print(f"Overall: {correct_q}/{total_q} = {overall:.4f} | {len(per_cell)}-Cell Macro: {macro:.4f}")
    print(f"Primitives: { {k: round(c / max(t, 1), 4) for k, (c, t) in prim_stats.items()} }")
    print(f"Joint Bundle Exact: {exact_bundles}/{total_bundles} = {exact_bundles / max(total_bundles, 1):.4f} | Contradictions: {contradicted_bundles}/{multi_q_bundles} = {contradicted_bundles / max(multi_q_bundles, 1):.4f}")
    print(f"banking77 overall: {cell_stats[('banking77', 'intent')][0]}/{cell_stats[('banking77', 'intent')][1]}")
    for var, (c, t) in sorted(bank_variant_stats.items()):
        print(f"  banking77 variant {var}: {c}/{t} = {c / max(t, 1):.4f}")
    for k, v in per_cell.items():
        print(f"  cell {k}: {v:.4f}")
    return {
        "overall_acc": overall, "macro_acc": macro, "num_cells": len(per_cell),
        "bundle_exact_acc": exact_bundles / max(total_bundles, 1),
        "contradiction_rate": contradicted_bundles / max(multi_q_bundles, 1),
        "per_cell": per_cell,
        "primitives": {k: c / max(t, 1) for k, (c, t) in prim_stats.items()},
        "banking77_variants": {k: c / max(t, 1) for k, (c, t) in bank_variant_stats.items()},
    }

hybrid_dev = hybrid_audit(dev_bundles, 'data/decision-v2-pilot/development.jsonl', "dev")
hybrid_test = hybrid_audit(test_bundles, 'data/decision-v2-pilot/test.jsonl', "test")

with open(f"{OUT_DIR}/witness-hybrid.json", "w") as f:
    json.dump({"dev": hybrid_dev, "test": hybrid_test}, f, indent=2)
print("\nSaved joint hybrid witness to witness-hybrid.json!")


Loaded per-option scorer from banking77-peroption-scorer.pt
dev: other 1060 bundles, banking77 116 bundles, variants Counter({'clean': 80, 'none_present': 12, 'none_absent': 12, 'permuted': 12})

=== HYBRID DEV ===
Overall: 1234/1440 = 0.8569 | 16-Cell Macro: 0.8553
Primitives: {'noul': 0.9405, 'choice': 0.8664, 'score': 0.6542}
Joint Bundle Exact: 985/1176 = 0.8376 | Contradictions: 1/172 = 0.0058
banking77 overall: 75/116
  banking77 variant clean: 57/80 = 0.7125
  banking77 variant none_absent: 0/12 = 0.0000
  banking77 variant none_present: 9/12 = 0.7500
  banking77 variant permuted: 9/12 = 0.7500
  cell agnews/is_business: 0.9333
  cell agnews/is_scitech: 0.9362
  cell agnews/is_sports: 0.9423
  cell agnews/is_world: 0.9500
  cell agnews/topic: 0.8793
  cell amazon/stars: 0.6125
  cell banking77/intent: 0.6466
  cell boolq/answer: 0.8875
  cell contrastive/decision: 1.0000
  cell dbpedia14/category: 0.9310
  cell imdb/positive: 0.9500
  cell mnli/relation: 0.8534
  cell sst5/senti

In [ ]:
# --- SYNTHETIC NONE TRAINING FOR BANKING77 (v2 scorer, pristine untouched) ---
import collections, copy, json, random, time, torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(123)
bank_train_all = [b for b in train_bundles if b.get('source') == 'banking77']
bank_dev = [b for b in dev_bundles if b.get('source') == 'banking77']
bank_test = [b for b in test_bundles if b.get('source') == 'banking77']

# 1. Build synthetic none bundles: remove the true intent, add none_of_these at a random position
synthetic_base = random.sample(bank_train_all, 300)
synthetic_bundles = []
for bundle in synthetic_base:
    q = bundle['questions'][0]
    gold_label = q['options'][int(q['gold'])]
    new_options = [o for o in q['options'] if o != gold_label]
    new_descs = [d for o, d in zip(q['options'], q['descs']) if o != gold_label]
    pos = random.randint(0, len(new_options))
    new_options.insert(pos, 'none_of_these')
    new_descs.insert(pos, None)
    synthetic_bundles.append({
        'source': 'banking77', 'state': bundle['state'],
        'questions': [{'qname': q['qname'], 'type': 'choice', 'instructions': q['instructions'],
                       'options': new_options, 'descs': new_descs, 'gold': pos}]
    })
print(f"synthetic none bundles: {len(synthetic_bundles)}")

def build_per_option_items(bundles):
    items = []
    for bi, bundle in enumerate(bundles):
        q = bundle['questions'][0]
        prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
        state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
        for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
            if not opt_ids:
                opt_ids = [tok.unk_token_id]
            state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
            ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
            items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2})
    return items

def precompute_hiddens(bundles, split_name):
    items = build_per_option_items(bundles)
    print(f"{split_name}: {len(items)} option encodings")
    bundle_hiddens = []
    bundle_golds = []
    for bundle in bundles:
        q = bundle['questions'][0]
        bundle_golds.append(int(q['gold']))
        bundle_hiddens.append(torch.zeros(len(q['options']), base.config.hidden_size))
    base.eval()
    with torch.no_grad():
        for start in range(0, len(items), 32):
            batch = items[start:start + 32]
            maxlen = max(len(x['ids']) for x in batch)
            ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
            attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
            mpos_t = torch.zeros(len(batch), dtype=torch.long)
            for i, x in enumerate(batch):
                ids_t[i, :len(x['ids'])] = torch.tensor(x['ids'])
                attn_t[i, :len(x['ids'])] = 1
                mpos_t[i] = x['mask']
            ids_t, attn_t = ids_t.to(device), attn_t.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
                h = out[torch.arange(len(batch), device=device), mpos_t.to(device)].float().cpu()
            for i, x in enumerate(batch):
                bundle_hiddens[x['bundle']][x['opt']] = h[i]
    return bundle_hiddens, bundle_golds

def load_variants(path):
    variants = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['_meta']['source'] == 'banking77':
                variants.append(d['_meta'].get('variant', 'unknown'))
    return variants

# 2. Reuse cached clean/dev/test hiddens if this is the same session, else precompute them
if 'train_h' in globals() and len(train_h) == 1000:
    clean_h, clean_g = train_h, train_g
    print("reusing cached clean train hiddens (1000 bundles)")
else:
    random.seed(42)
    clean_sample = random.sample(bank_train_all, 1000)
    clean_h, clean_g = precompute_hiddens(clean_sample, "clean train")

syn_h, syn_g = precompute_hiddens(synthetic_bundles, "synthetic none")

if 'dev_h' in globals() and len(dev_h) == 116:
    dev_h_c, dev_g_c, test_h_c, test_g_c = dev_h, dev_g, test_h, test_g
    print("reusing cached dev/test hiddens")
else:
    dev_h_c, dev_g_c = precompute_hiddens(bank_dev, "dev")
    test_h_c, test_g_c = precompute_hiddens(bank_test, "test")

dev_variants = load_variants('data/decision-v2-pilot/development.jsonl')
test_variants = load_variants('data/decision-v2-pilot/test.jsonl')

def eval_variant(hiddens, golds, variants, scorer):
    scorer.eval()
    correct = 0
    var_stats = collections.defaultdict(lambda: [0, 0])
    with torch.no_grad():
        for h, gold, var in zip(hiddens, golds, variants):
            pred = int(scorer(h.to(device)).squeeze(-1).argmax().item())
            ok = int(pred == gold)
            correct += ok
            var_stats[var][0] += ok
            var_stats[var][1] += 1
    return correct / len(golds), {k: f"{c}/{t} = {c / max(t, 1):.4f}" for k, (c, t) in sorted(var_stats.items())}

# 3. Start from v1 scorer, fine-tune on clean + synthetic none
scorer_v2 = copy.deepcopy(model.scorer).to(device)
_v1 = torch.load(f"{OUT_DIR}/banking77-peroption-scorer.pt", map_location=device)
_v1_state = _v1['state'] if isinstance(_v1, dict) and 'state' in _v1 else _v1
scorer_v2.load_state_dict(_v1_state)
print("Loaded v1 scorer as starting point")

combined_h = clean_h + syn_h
combined_g = clean_g + syn_g
opt = torch.optim.AdamW(scorer_v2.parameters(), lr=2e-4, weight_decay=0.01)
best_dev = 0.0
best_state = None
for epoch in range(3):
    scorer_v2.train()
    order = list(range(len(combined_h)))
    random.shuffle(order)
    tot_loss, n = 0.0, 0
    for start in range(0, len(order), 32):
        idx = order[start:start + 32]
        h_batch = torch.stack([combined_h[i] for i in idx]).to(device)
        g_batch = torch.tensor([combined_g[i] for i in idx], dtype=torch.long, device=device)
        loss = F.cross_entropy(scorer_v2(h_batch).squeeze(-1), g_batch)
        opt.zero_grad()
        loss.backward()
        opt.step()
        tot_loss += float(loss.item())
        n += 1
    dev_acc, dev_var = eval_variant(dev_h_c, dev_g_c, dev_variants, scorer_v2)
    print(f"epoch {epoch + 1}/3 loss {tot_loss / max(n, 1):.4f} | dev banking77: {dev_acc:.4f} | dev variants: {dev_var}")
    if dev_acc > best_dev:
        best_dev = dev_acc
        best_state = copy.deepcopy(scorer_v2.state_dict())

scorer_v2.load_state_dict(best_state)
test_acc, test_var = eval_variant(test_h_c, test_g_c, test_variants, scorer_v2)
print(f"\nBEST dev banking77: {best_dev:.4f}")
print(f"FINAL test banking77 (v2): {test_acc:.4f} | test variants: {test_var}")
torch.save({'state': best_state, 'dev_acc': best_dev, 'test_acc': test_acc}, f"{OUT_DIR}/banking77-peroption-scorer-v2.pt")
print("Saved v2 scorer separately (v1 and pristine untouched)")


synthetic none bundles: 300
reusing cached clean train hiddens (1000 bundles)
synthetic none: 23100 option encodings
reusing cached dev/test hiddens
Loaded v1 scorer as starting point
epoch 1/3 loss 1.8976 | dev banking77: 0.6379 | dev variants: {'clean': '56/80 = 0.7000', 'none_absent': '0/12 = 0.0000', 'none_present': '9/12 = 0.7500', 'permuted': '9/12 = 0.7500'}
epoch 2/3 loss 1.4046 | dev banking77: 0.6379 | dev variants: {'clean': '56/80 = 0.7000', 'none_absent': '0/12 = 0.0000', 'none_present': '9/12 = 0.7500', 'permuted': '9/12 = 0.7500'}
epoch 3/3 loss 1.2507 | dev banking77: 0.6552 | dev variants: {'clean': '57/80 = 0.7125', 'none_absent': '1/12 = 0.0833', 'none_present': '9/12 = 0.7500', 'permuted': '9/12 = 0.7500'}

BEST dev banking77: 0.6552
FINAL test banking77 (v2): 0.4914 | test variants: {'clean': '49/80 = 0.6125', 'none_absent': '0/12 = 0.0000', 'none_present': '4/12 = 0.3333', 'permuted': '4/12 = 0.3333'}
Saved v2 scorer separately (v1 and pristine untouched)


In [ ]:
# --- THRESHOLD ABSTENTION PROBE FOR BANKING77 NONE (no retraining, cached hiddens) ---
import collections, copy, json, torch
import torch.nn.functional as F

# reload v2 scorer (works even if you restarted, after the restore cell)
scorer_t = copy.deepcopy(model.scorer).to(device)
_v2 = torch.load(f"{OUT_DIR}/banking77-peroption-scorer-v2.pt", map_location=device)
_v2_state = _v2['state'] if isinstance(_v2, dict) and 'state' in _v2 else _v2
scorer_t.load_state_dict(_v2_state)
scorer_t.eval()

def load_variants(path):
    variants = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['_meta']['source'] == 'banking77':
                variants.append(d['_meta'].get('variant', 'unknown'))
    return variants

def score_bundles(hiddens, bundles):
    # returns per-bundle: (best_real_idx, p_best_real, pred_all_idx, gold, options)
    results = []
    scorer_t.eval()
    with torch.no_grad():
        for h, bundle in zip(hiddens, bundles):
            q = bundle['questions'][0]
            opts = q['options']
            s = scorer_t(h.to(device)).squeeze(-1).float()
            none_idx = opts.index('none_of_these') if 'none_of_these' in opts else -1
            real_idx = [i for i in range(len(opts)) if i != none_idx]
            real_scores = s[real_idx]
            p_best_real = float(F.softmax(real_scores, dim=-1).max().item())
            best_real = int(real_idx[int(real_scores.argmax().item())])
            pred_all = int(s.argmax().item())
            results.append((best_real, p_best_real, pred_all, int(q['gold']), opts, none_idx))
    return results

def eval_with_threshold(results, variants, T):
    correct = 0
    var_stats = collections.defaultdict(lambda: [0, 0])
    for (best_real, p_best, pred_all, gold, opts, none_idx), var in zip(results, variants):
        if none_idx >= 0 and p_best < T:
            pred = none_idx
        else:
            pred = best_real
        ok = int(pred == gold)
        correct += ok
        var_stats[var][0] += ok
        var_stats[var][1] += 1
    return correct / len(results), {k: f"{c}/{t}" for k, (c, t) in sorted(var_stats.items())}

bank_dev = [b for b in dev_bundles if b.get('source') == 'banking77']
bank_test = [b for b in test_bundles if b.get('source') == 'banking77']
dev_variants = load_variants('data/decision-v2-pilot/development.jsonl')
test_variants = load_variants('data/decision-v2-pilot/test.jsonl')

# dev_h / test_h are the cached hiddens from the training cell; if you restarted, re-run that precompute first
dev_res = score_bundles(dev_h, bank_dev)
test_res = score_bundles(test_h, bank_test)

# baseline: argmax over all options (should match v2: dev 0.6552, test 0.4914)
dev_base = sum(1 for (_, _, pred_all, gold, _, _) in dev_res if pred_all == gold) / len(dev_res)
test_base = sum(1 for (_, _, pred_all, gold, _, _) in test_res if pred_all == gold) / len(test_res)
print(f"argmax-all baseline: dev {dev_base:.4f} | test {test_base:.4f}")

print("\nThreshold sweep on DEV (p_best_real < T -> predict none, if none option present):")
best_T, best_dev = 0.0, 0.0
for T in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    acc, var = eval_with_threshold(dev_res, dev_variants, T)
    print(f"  T={T:.2f} dev {acc:.4f} variants {var}")
    if acc > best_dev:
        best_dev, best_T = acc, T

test_acc, test_var = eval_with_threshold(test_res, test_variants, best_T)
print(f"\nBest T from dev: {best_T:.2f} (dev {best_dev:.4f})")
print(f"Test with best T: {test_acc:.4f} variants {test_var}")
print("If test none_absent stays at 0/12 here too, threshold abstention does not separate none either, and we document none_absent as a limitation and move to writing.")


argmax-all baseline: dev 0.6552 | test 0.4914

Threshold sweep on DEV (p_best_real < T -> predict none, if none option present):
  T=0.00 dev 0.6466 variants {'clean': '57/80', 'none_absent': '0/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.05 dev 0.6466 variants {'clean': '57/80', 'none_absent': '0/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.10 dev 0.6552 variants {'clean': '57/80', 'none_absent': '1/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.15 dev 0.6897 variants {'clean': '57/80', 'none_absent': '5/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.20 dev 0.7155 variants {'clean': '57/80', 'none_absent': '8/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.25 dev 0.7241 variants {'clean': '57/80', 'none_absent': '9/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.30 dev 0.7241 variants {'clean': '57/80', 'none_absent': '9/12', 'none_present': '9/12', 'permuted': '9/12'}
  T=0.40 dev 0.7328 variants {'clean': '57/80', 'none_absent': '10

In [ ]:
# --- FINAL HYBRID AUDIT v3: pristine packed + v2 per-option banking77 + T=0.40 threshold ---
import collections, copy, json, torch
import torch.nn.functional as F

peroption_scorer = copy.deepcopy(model.scorer).to(device)
_v2 = torch.load(f"{OUT_DIR}/banking77-peroption-scorer-v2.pt", map_location=device)
_v2_state = _v2['state'] if isinstance(_v2, dict) and 'state' in _v2 else _v2
peroption_scorer.load_state_dict(_v2_state)
peroption_scorer.eval()
model.eval()
base.eval()
T_THRESHOLD = 0.40
print(f"Loaded v2 scorer, threshold T={T_THRESHOLD}")

def load_variants(path):
    variants = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['_meta']['source'] == 'banking77':
                variants.append(d['_meta'].get('variant', 'unknown'))
    return variants

def hybrid_audit_v3(bundles_raw, variant_path, split_name):
    variants = load_variants(variant_path)
    bank_bundles = [b for b in bundles_raw if b.get('source') == 'banking77']
    other_bundles = [b for b in bundles_raw if b.get('source') != 'banking77']
    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q_bundles, contradicted_bundles = 0, 0
    cell_stats = collections.defaultdict(lambda: [0, 0])
    prim_stats = collections.defaultdict(lambda: [0, 0])
    bank_variant_stats = collections.defaultdict(lambda: [0, 0])

    # Phase A: pristine packed, non-banking77 (unchanged)
    built = [build_bundle_examples(x) for x in other_bundles]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    with torch.no_grad():
        for ids, attn, mpos, omask, golds, raw_b in dl:
            ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(base, ids, attn, mpos, omask, use_graph=True)
            for b_idx, b in enumerate(raw_b):
                qs = b['questions']
                b_all_correct = True
                preds_by_type = {}
                for j, q in enumerate(qs):
                    n_opt = int(q['n_opt'])
                    pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                    gold = int(golds[b_idx][j])
                    ok = int(pred == gold)
                    correct_q += ok; total_q += 1
                    if not ok: b_all_correct = False
                    prim_stats[q['type']][0] += ok; prim_stats[q['type']][1] += 1
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][0] += ok
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][1] += 1
                    preds_by_type[q['type']] = (pred, q)
                total_bundles += 1
                if b_all_correct: exact_bundles += 1
                if len(qs) > 1:
                    multi_q_bundles += 1
                    if 'choice' in preds_by_type and 'noul' in preds_by_type:
                        c_pred, c_q = preds_by_type['choice']
                        n_pred, n_q = preds_by_type['noul']
                        c_opts = c_q.get('options', [])
                        if c_pred < len(c_opts):
                            if str(c_opts[c_pred]).lower() in str(n_q.get('instructions', '')).lower() and n_pred == 0:
                                contradicted_bundles += 1

    # Phase B: per-option v2 scorer + threshold, banking77 only
    bank_items = []
    for bi, bundle in enumerate(bank_bundles):
        q = bundle['questions'][0]
        prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
        state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
        for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d: opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
            if not opt_ids: opt_ids = [tok.unk_token_id]
            state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
            ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
            bank_items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2})
    bank_scores = [0.0] * len(bank_items)
    with torch.no_grad():
        for start in range(0, len(bank_items), 32):
            batch = bank_items[start:start + 32]
            maxlen = max(len(x['ids']) for x in batch)
            ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
            attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
            mpos_t = torch.zeros(len(batch), dtype=torch.long)
            for i, x in enumerate(batch):
                ids_t[i, :len(x['ids'])] = torch.tensor(x['ids']); attn_t[i, :len(x['ids'])] = 1; mpos_t[i] = x['mask']
            ids_t, attn_t = ids_t.to(device), attn_t.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
                h = out[torch.arange(len(batch), device=device), mpos_t.to(device)]
                s = peroption_scorer(h).squeeze(-1).float()
            for i, x in enumerate(batch):
                bank_scores[start + i] = float(s[i].item())
    bank_idx = 0
    for bi, bundle in enumerate(bank_bundles):
        q = bundle['questions'][0]
        opts = q['options']
        n = len(opts)
        bundle_scores = torch.tensor(bank_scores[bank_idx:bank_idx + n]); bank_idx += n
        none_idx = opts.index('none_of_these') if 'none_of_these' in opts else -1
        real_idx = [i for i in range(n) if i != none_idx]
        real_scores = bundle_scores[real_idx]
        p_best_real = float(F.softmax(real_scores, dim=-1).max().item())
        best_real = int(real_idx[int(real_scores.argmax().item())])
        if none_idx >= 0 and p_best_real < T_THRESHOLD:
            pred = none_idx
        else:
            pred = best_real
        gold = int(q['gold']); ok = int(pred == gold)
        correct_q += ok; total_q += 1; total_bundles += 1
        if ok: exact_bundles += 1
        prim_stats['choice'][0] += ok; prim_stats['choice'][1] += 1
        cell_stats[('banking77', 'intent')][0] += ok; cell_stats[('banking77', 'intent')][1] += 1
        var = variants[bi] if bi < len(variants) else 'unknown'
        bank_variant_stats[var][0] += ok; bank_variant_stats[var][1] += 1

    overall = correct_q / max(total_q, 1)
    per_cell = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro = sum(per_cell.values()) / max(len(per_cell), 1)
    print(f"\n=== HYBRID v3 {split_name.upper()} ===")
    print(f"Overall: {correct_q}/{total_q} = {overall:.4f} | {len(per_cell)}-Cell Macro: {macro:.4f}")
    print(f"Primitives: { {k: round(c / max(t, 1), 4) for k, (c, t) in prim_stats.items()} }")
    print(f"Joint: {exact_bundles}/{total_bundles} = {exact_bundles / max(total_bundles, 1):.4f} | Contradictions: {contradicted_bundles}/{multi_q_bundles}")
    print(f"banking77: {cell_stats[('banking77', 'intent')]} variants: { {k: f'{c}/{t}' for k, (c, t) in sorted(bank_variant_stats.items())} }")
    return {"overall_acc": overall, "macro_acc": macro, "bundle_exact_acc": exact_bundles / max(total_bundles, 1),
            "contradiction_rate": contradicted_bundles / max(multi_q_bundles, 1), "per_cell": per_cell,
            "primitives": {k: c / max(t, 1) for k, (c, t) in prim_stats.items()},
            "banking77_variants": {k: c / max(t, 1) for k, (c, t) in bank_variant_stats.items()}}

hybrid_v3_dev = hybrid_audit_v3(dev_bundles, 'data/decision-v2-pilot/development.jsonl', "dev")
hybrid_v3_test = hybrid_audit_v3(test_bundles, 'data/decision-v2-pilot/test.jsonl', "test")
with open(f"{OUT_DIR}/witness-hybrid-v3.json", "w") as f:
    json.dump({"dev": hybrid_v3_dev, "test": hybrid_v3_test, "threshold_T": T_THRESHOLD}, f, indent=2)
print("\nSaved witness-hybrid-v3.json!")


Loaded v2 scorer, threshold T=0.4

=== HYBRID v3 DEV ===
Overall: 1244/1440 = 0.8639 | 16-Cell Macro: 0.8607
Primitives: {'noul': 0.9405, 'choice': 0.8807, 'score': 0.6542}
Joint: 995/1176 = 0.8461 | Contradictions: 1/172
banking77: [85, 116] variants: {'clean': '57/80', 'none_absent': '10/12', 'none_present': '9/12', 'permuted': '9/12'}

=== HYBRID v3 TEST ===
Overall: 1223/1440 = 0.8493 | 16-Cell Macro: 0.8502
Primitives: {'noul': 0.9306, 'choice': 0.8649, 'score': 0.6333}
Joint: 970/1176 = 0.8248 | Contradictions: 0/172
banking77: [62, 116] variants: {'clean': '49/80', 'none_absent': '6/12', 'none_present': '3/12', 'permuted': '4/12'}

Saved witness-hybrid-v3.json!


In [ ]:
# --- GRAPH ABLATION: pristine v5, graph ON vs OFF (no retraining) ---
import collections, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

print(f"graph_gate value: {float(model.graph_gate.item()):.6f} | tanh(gate): {float(torch.tanh(model.graph_gate).item()):.6f}")
print("If tanh(gate) is 0, graph-on and graph-off MUST be identical for pristine v5.")
model.eval(); base.eval()

def ablation_audit(bundles_raw, split_name, use_graph):
    other_bundles = [b for b in bundles_raw if b.get('source') != 'banking77']
    built = [build_bundle_examples(x) for x in other_bundles]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q, contradicted = 0, 0
    with torch.no_grad():
        for ids, attn, mpos, omask, golds, raw_b in dl:
            ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(base, ids, attn, mpos, omask, use_graph=use_graph)
            for b_idx, b in enumerate(raw_b):
                qs = b['questions']
                b_all = True
                preds = {}
                for j, q in enumerate(qs):
                    n_opt = int(q['n_opt'])
                    pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                    gold = int(golds[b_idx][j]); ok = int(pred == gold)
                    correct_q += ok; total_q += 1
                    if not ok: b_all = False
                    preds[q['type']] = (pred, q)
                total_bundles += 1
                if b_all: exact_bundles += 1
                if len(qs) > 1:
                    multi_q += 1
                    if 'choice' in preds and 'noul' in preds:
                        c_pred, c_q = preds['choice']; n_pred, n_q = preds['noul']
                        c_opts = c_q.get('options', [])
                        if c_pred < len(c_opts):
                            if str(c_opts[c_pred]).lower() in str(n_q.get('instructions', '')).lower() and n_pred == 0:
                                contradicted += 1
    print(f"{split_name} graph={'ON' if use_graph else 'OFF'}: overall {correct_q}/{total_q} = {correct_q / max(total_q, 1):.4f} | joint {exact_bundles}/{total_bundles} = {exact_bundles / max(total_bundles, 1):.4f} | contradictions {contradicted}/{multi_q}")
    return correct_q, total_q, exact_bundles, total_bundles, contradicted, multi_q

for split_name, bundles in [("dev", dev_bundles), ("test", test_bundles)]:
    ablation_audit(bundles, split_name, use_graph=True)
    ablation_audit(bundles, split_name, use_graph=False)
print("\nDone. Share all 4 lines.")


graph_gate value: 0.000000 | tanh(gate): 0.000000
If tanh(gate) is 0, graph-on and graph-off MUST be identical for pristine v5.
dev graph=ON: overall 1159/1324 = 0.8754 | joint 910/1060 = 0.8585 | contradictions 1/172
dev graph=OFF: overall 1159/1324 = 0.8754 | joint 910/1060 = 0.8585 | contradictions 1/172
test graph=ON: overall 1161/1324 = 0.8769 | joint 908/1060 = 0.8566 | contradictions 0/172
test graph=OFF: overall 1161/1324 = 0.8769 | joint 908/1060 = 0.8566 | contradictions 0/172

Done. Share all 4 lines.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- INSTANT 2-SECOND MERGE INTO ONE FILE: myna-v5-unified.pt ---
import json, torch

OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"

# 1. Load your saved packed head and decoupled head
v5_ckpt = torch.load(f"{OUT_DIR}/myna-v5-best.pt", map_location="cpu")
v2_ckpt = torch.load(f"{OUT_DIR}/banking77-peroption-scorer-v2.pt", map_location="cpu")

packed_state = {k.replace("scorer.", ""): v for k, v in v5_ckpt["state"].items() if k.startswith("scorer.")}
decoupled_state = v2_ckpt["state"] if isinstance(v2_ckpt, dict) and "state" in v2_ckpt else v2_ckpt

with open(f"{OUT_DIR}/witness-hybrid-v3.json") as f:
    hybrid_v3 = json.load(f)
with open(f"{OUT_DIR}/witness-v5-master-paper.json") as f:
    master_v5 = json.load(f)

# 2. Combine into one single unified checkpoint dictionary
unified_state_dict = {}
for k, v in packed_state.items():
    unified_state_dict[f"packed_scorer.{k}"] = v
for k, v in decoupled_state.items():
    unified_state_dict[f"decoupled_scorer.{k}"] = v

unified_payload = {
    "model_label": "myna-v5-unified (DeBERTa-v3-large + Cardinality-Routed Readout Head)",
    "state_dict": unified_state_dict,
    "routing_rule": "len(options) <= 16 -> packed_scorer | len(options) > 16 -> decoupled_scorer (T=0.40)",
    "max_packed_options": 16,
    "none_threshold": 0.40,
    "temperatures": master_v5.get("temperatures", {"choice": 1.55, "score": 1.4, "noul": 1.6}),
    "pristine_packed_audit": {
        "dev": master_v5["dev_16cell_audit"],
        "test": master_v5["test_16cell_audit"],
        "calibration_and_g5": master_v5["calibration_and_g5"]
    },
    "unified_hybrid_v3_audit": hybrid_v3
}

out_path = f"{OUT_DIR}/myna-v5-unified.pt"
torch.save(unified_payload, out_path)
print("SUCCESS! Created single unified checkpoint:", out_path)

SUCCESS! Created single unified checkpoint: /content/drive/MyDrive/myna-v5-prototype/myna-v5-unified.pt


In [3]:
# --- FINAL VERIFICATION CELL IN NEW ACCOUNT ---
import os, json, torch
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"
unified_path = f"{OUT_DIR}/myna-v5-unified.pt"

# 2. Verify the unified checkpoint loads successfully
if os.path.exists(unified_path):
    ckpt = torch.load(unified_path, map_location="cpu")
    print("SUCCESS: Loaded myna-v5-unified.pt!")
    print(f"Model Label: {ckpt.get('model_label', 'N/A')}")
    print(f"Routing Rule: {ckpt.get('routing_rule', 'N/A')}")
    print(f"Test Accuracy from Master Witness: {ckpt.get('pristine_packed_audit', {}).get('test', {}).get('overall_acc', 'N/A')}")
else:
    print(f"ERROR: Checkpoint not found at {unified_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SUCCESS: Loaded myna-v5-unified.pt!
Model Label: myna-v5-unified (DeBERTa-v3-large + Cardinality-Routed Readout Head)
Routing Rule: len(options) <= 16 -> packed_scorer | len(options) > 16 -> decoupled_scorer (T=0.40)
Test Accuracy from Master Witness: 0.8069444444444445


In [4]:
# --- INSPECT UNIFIED CHECKPOINT (CPU only, no evaluation yet) ---
import os, torch
from google.colab import drive

drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"

print("Folder contents:")
for name in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, name)
    if os.path.isfile(p):
        print(f"  {name}  {os.path.getsize(p) / 1024 / 1024:.1f} MB")

unified_path = f"{OUT_DIR}/myna-v5-unified.pt"
print(f"\nUnified size: {os.path.getsize(unified_path) / 1024 / 1024:.1f} MB")
ckpt = torch.load(unified_path, map_location="cpu")
print(f"Unified type: {type(ckpt)}")
if isinstance(ckpt, dict):
    for k, v in ckpt.items():
        if hasattr(v, 'shape'):
            print(f"  {k}: tensor shape {tuple(v.shape)} dtype {v.dtype}")
        elif isinstance(v, dict):
            print(f"  {k}: dict with keys {list(v.keys())[:20]}")
        elif isinstance(v, (list, tuple)):
            print(f"  {k}: list/tuple len {len(v)}")
        else:
            print(f"  {k}: {repr(v)[:300]}")
else:
    print("Unified checkpoint is not a dict - share its type and structure.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder contents:
  banking77-peroption-scorer-v2.pt  2.0 MB
  banking77-peroption-scorer.pt  2.0 MB
  myna-v5-best.pt  2.0 MB
  myna-v5-unified.pt  4.0 MB
  witness-hybrid-v3.json  0.0 MB
  witness-hybrid.json  0.0 MB
  witness-v5-full.json  0.0 MB
  witness-v5-master-paper.json  0.0 MB
  witness-v5.1-paper.json  0.0 MB
  witness-v5.json  0.0 MB

Unified size: 4.0 MB
Unified type: <class 'dict'>
  model_label: 'myna-v5-unified (DeBERTa-v3-large + Cardinality-Routed Readout Head)'
  state_dict: dict with keys ['packed_scorer.0.weight', 'packed_scorer.0.bias', 'packed_scorer.1.weight', 'packed_scorer.1.bias', 'packed_scorer.4.weight', 'packed_scorer.4.bias', 'decoupled_scorer.0.weight', 'decoupled_scorer.0.bias', 'decoupled_scorer.1.weight', 'decoupled_scorer.1.bias', 'decoupled_scorer.4.weight', 'decoupled_scorer.4.bias']
  routing_rule: 'len(options) <= 16 ->

In [5]:
# --- QUICK CHECK: backbone folder + stored audit numbers inside unified ---
import os, json, torch
from google.colab import drive

OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"
print("Folders in prototype dir:")
for name in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, name)
    if os.path.isdir(p):
        print(f"  {name}/")
        for sub in sorted(os.listdir(p))[:20]:
            sp = os.path.join(p, sub)
            if os.path.isfile(sp):
                print(f"    {sub}  {os.path.getsize(sp) / 1024 / 1024:.1f} MB")

ckpt = torch.load(f"{OUT_DIR}/myna-v5-unified.pt", map_location="cpu")
print(f"\nTemperatures: {ckpt.get('temperatures')}")
print(f"Routing: {ckpt.get('routing_rule')} | max_packed: {ckpt.get('max_packed_options')} | threshold: {ckpt.get('none_threshold')}")
for audit_name in ['pristine_packed_audit', 'unified_hybrid_v3_audit']:
    a = ckpt.get(audit_name, {})
    print(f"\n{audit_name}:")
    for split in ['dev', 'test']:
        s = a.get(split, {})
        print(f"  {split}: overall {s.get('overall_acc')} | macro {s.get('macro_acc')} | joint {s.get('bundle_exact_acc')} | contradictions {s.get('contradiction_rate')}")


Folders in prototype dir:
  deberta-v5-finetuned/
    config.json  0.0 MB
    model.safetensors  1655.7 MB
    tokenizer.json  8.0 MB
    tokenizer_config.json  0.0 MB

Temperatures: {'choice': 1.25, 'score': 1.0, 'noul': 1.0}
Routing: len(options) <= 16 -> packed_scorer | len(options) > 16 -> decoupled_scorer (T=0.40) | max_packed: 16 | threshold: 0.4

pristine_packed_audit:
  dev: overall 0.80625 | macro 0.8159860472280226 | joint 0.7755102040816326 | contradictions 0.005813953488372093
  test: overall 0.8069444444444445 | macro 0.817302742629012 | joint 0.7729591836734694 | contradictions 0.0

unified_hybrid_v3_audit:
  dev: overall 0.8638888888888889 | macro 0.8607058748142296 | joint 0.8460884353741497 | contradictions 0.005813953488372093
  test: overall 0.8493055555555555 | macro 0.8501691219393568 | joint 0.8248299319727891 | contradictions 0.0


In [7]:
# --- SETUP ONLY: define device, backbone, model for unified reproduction (no training, no Stage 3) ---
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/content/drive/MyDrive/myna-v5-prototype"
print(f"device: {device} | OUT_DIR: {OUT_DIR}")

tok = AutoTokenizer.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned")
base = AutoModel.from_pretrained(f"{OUT_DIR}/deberta-v5-finetuned", torch_dtype=torch.float32).to(device)
MASK_ID = tok.mask_token_id; CLS_ID = tok.cls_token_id; SEP_ID = tok.sep_token_id; PAD_ID = tok.pad_token_id

def text_of(x):
    if isinstance(x, str): return x
    if x is None: return ''
    if isinstance(x, dict): return ' | '.join(f"{k}: {text_of(v)}" for k, v in x.items())
    if isinstance(x, list): return ' '.join(text_of(v) for v in x)
    return str(x)

def build_bundle_examples(bundle):
    state_text = text_of(bundle['state'])
    encoded_qs = []
    for q in bundle['questions']:
        n_opt = len(q['options'])
        max_opt_toks = max(2, (MAX_LEN - 96) // max(1, n_opt) - 2)
        opt_chunks = []
        for label, desc in zip(q['options'], q['descs']):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d and max_opt_toks > 4:
                opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids'][:max_opt_toks]
            if not opt_ids: opt_ids = [tok.unk_token_id]
            opt_chunks.append(opt_ids + [MASK_ID, SEP_ID])
        prefix = f"Task ({bundle['source']} / {q['qname']}): {text_of(q['instructions'])} | Context: "
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids'][:64]
        state_ids = tok(state_text, add_special_tokens=False)['input_ids'][: (MAX_LEN - 64 - sum(len(c) for c in opt_chunks))]
        ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID]
        mask_pos = []
        for chunk in opt_chunks:
            if len(ids) + len(chunk) > MAX_LEN: break
            ids += chunk
            mask_pos.append(len(ids) - 2)
        encoded_qs.append({'qname': q['qname'], 'type': q['type'], 'options': q['options'],
                           'instructions': q['instructions'], 'input_ids': ids, 'mask_pos': mask_pos,
                           'gold': int(q['gold']), 'n_opt': len(mask_pos)})
    return {'source': bundle['source'], 'questions': encoded_qs}

def bundle_collate(bundle_list):
    max_q = max(len(b['questions']) for b in bundle_list)
    max_len = max(max(len(q['input_ids']) for q in b['questions']) for b in bundle_list)
    max_opt = max(max(len(q['mask_pos']) for q in b['questions']) for b in bundle_list)
    B = len(bundle_list)
    ids = torch.full((B, max_q, max_len), PAD_ID, dtype=torch.long)
    attn = torch.zeros((B, max_q, max_len), dtype=torch.long)
    mpos = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    omask = torch.zeros((B, max_q, max_opt), dtype=torch.long)
    golds = []
    for i, b in enumerate(bundle_list):
        g_list = []
        for j, q in enumerate(b['questions']):
            ids[i, j, :len(q['input_ids'])] = torch.tensor(q['input_ids'])
            attn[i, j, :len(q['input_ids'])] = 1
            mpos[i, j, :len(q['mask_pos'])] = torch.tensor(q['mask_pos'])
            omask[i, j, :len(q['mask_pos'])] = 1
            g_list.append(int(q['gold']))
        golds.append(g_list)
    return ids, attn, mpos, omask, golds, bundle_list

class QuestionGraphHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )
        self.q_norm = nn.LayerNorm(hidden_size)
        self.q_attn = nn.MultiheadAttention(hidden_size, num_heads=8, dropout=0.1, batch_first=True)
        self.graph_gate = nn.Parameter(torch.zeros(1))
    def forward(self, base_model, input_ids, attn, mask_pos, opt_mask, use_graph=True):
        B, M, L = input_ids.shape
        flat_ids = input_ids.view(B * M, L); flat_attn = attn.view(B * M, L)
        out = base_model(input_ids=flat_ids, attention_mask=flat_attn).last_hidden_state
        _, _, O = mask_pos.shape; flat_mpos = mask_pos.view(B * M, O)
        idx = torch.arange(B * M, device=input_ids.device).unsqueeze(1)
        h_options = out[idx, flat_mpos].view(B, M, O, -1)
        if use_graph and M > 1:
            flat_ho = h_options.view(B, M * O, -1)
            flat_mask = (opt_mask.view(B, M * O) == 0)
            norm_ho = self.q_norm(flat_ho)
            graph_out, _ = self.q_attn(norm_ho, norm_ho, norm_ho, key_padding_mask=flat_mask)
            h_options = h_options + torch.tanh(self.graph_gate) * graph_out.view(B, M, O, -1)
        logits = self.scorer(h_options).squeeze(-1).float()
        logits = logits.masked_fill(opt_mask.view(B, M, O) == 0, -1e9)
        return logits

model = QuestionGraphHead(base.config.hidden_size).to(device)
print("Setup done. Do NOT load myna-v5-best.pt here - the reproduction cell will overwrite the scorer with unified weights.")


device: cuda | OUT_DIR: /content/drive/MyDrive/myna-v5-prototype


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Setup done. Do NOT load myna-v5-best.pt here - the reproduction cell will overwrite the scorer with unified weights.


In [15]:
# --- FRESH REPRODUCTION FROM UNIFIED ONLY (count routing, T=0.40) ---
import collections, copy, json, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

unified = torch.load(f"{OUT_DIR}/myna-v5-unified.pt", map_location=device)
print(f"Unified label: {unified.get('model_label')}")
print(f"Routing: {unified.get('routing_rule')}")

packed_state = {k.replace('packed_scorer.', '', 1): v for k, v in unified['state_dict'].items() if k.startswith('packed_scorer.')}
decoupled_state = {k.replace('decoupled_scorer.', '', 1): v for k, v in unified['state_dict'].items() if k.startswith('decoupled_scorer.')}
print(f"Packed scorer keys: {sorted(packed_state.keys())}")
print(f"Decoupled scorer keys: {sorted(decoupled_state.keys())}")
model.scorer.load_state_dict(packed_state)
peroption_scorer = copy.deepcopy(model.scorer).to(device)
peroption_scorer.load_state_dict(decoupled_state)
peroption_scorer.eval(); model.eval(); base.eval()
T_THRESHOLD = float(unified.get('none_threshold', 0.40))
print(f"Using unified scorers only, threshold T={T_THRESHOLD}")

def load_variants(path):
    variants = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            if d['_meta']['source'] == 'banking77':
                variants.append(d['_meta'].get('variant', 'unknown'))
    return variants

def max_opts(bundle):
    return max(len(q['options']) for q in bundle['questions'])

def repro_audit(bundles_raw, variant_path, split_name):
    variants = load_variants(variant_path)
    decoupled_bundles = [b for b in bundles_raw if max_opts(b) > 16]
    other_bundles = [b for b in bundles_raw if max_opts(b) <= 16]
    print(f"{split_name}: packed (<=16 opts) {len(other_bundles)} bundles, decoupled (>16 opts) {len(decoupled_bundles)} bundles, variants {collections.Counter(variants)}")
    total_q, correct_q = 0, 0
    total_bundles, exact_bundles = 0, 0
    multi_q, contradicted = 0, 0
    cell_stats = collections.defaultdict(lambda: [0, 0])
    prim_stats = collections.defaultdict(lambda: [0, 0])
    bank_var_stats = collections.defaultdict(lambda: [0, 0])

    built = [build_bundle_examples(x) for x in other_bundles]
    dl = DataLoader(built, batch_size=4, shuffle=False, collate_fn=bundle_collate)
    with torch.no_grad():
        for ids, attn, mpos, omask, golds, raw_b in dl:
            ids, attn, mpos, omask = ids.to(device), attn.to(device), mpos.to(device), omask.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                logits = model(base, ids, attn, mpos, omask, use_graph=True)
            for b_idx, b in enumerate(raw_b):
                qs = b['questions']; b_all = True; preds = {}
                for j, q in enumerate(qs):
                    n_opt = int(q['n_opt'])
                    pred = int(logits[b_idx, j, :n_opt].argmax(dim=-1).item())
                    gold = int(golds[b_idx][j]); ok = int(pred == gold)
                    correct_q += ok; total_q += 1
                    if not ok: b_all = False
                    prim_stats[q['type']][0] += ok; prim_stats[q['type']][1] += 1
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][0] += ok
                    cell_stats[(b.get('source', 'unknown'), q['qname'])][1] += 1
                    preds[q['type']] = (pred, q)
                total_bundles += 1
                if b_all: exact_bundles += 1
                if len(qs) > 1:
                    multi_q += 1
                    if 'choice' in preds and 'noul' in preds:
                        c_pred, c_q = preds['choice']; n_pred, n_q = preds['noul']
                        c_opts = c_q.get('options', [])
                        if c_pred < len(c_opts):
                            if str(c_opts[c_pred]).lower() in str(n_q.get('instructions', '')).lower() and n_pred == 0:
                                contradicted += 1

    bank_items = []
    for bi, bundle in enumerate(decoupled_bundles):
        q = bundle['questions'][0]
        prefix_ids = tok(f"Task (banking77 / intent): {text_of(q['instructions'])} | Context: ", add_special_tokens=False)['input_ids'][:64]
        state_all = tok(text_of(bundle['state']), add_special_tokens=False)['input_ids']
        for oi, (label, desc) in enumerate(zip(q['options'], q['descs'])):
            opt_text = label.replace('_', ' ')
            d = text_of(desc)
            if d: opt_text += ': ' + ' '.join(d.split()[:10])
            opt_ids = tok(opt_text, add_special_tokens=False)['input_ids']
            if not opt_ids: opt_ids = [tok.unk_token_id]
            state_ids = state_all[:max(0, MAX_LEN - len(prefix_ids) - len(opt_ids) - 4)]
            ids = [CLS_ID] + prefix_ids + state_ids + [SEP_ID] + opt_ids + [MASK_ID, SEP_ID]
            bank_items.append({'bundle': bi, 'opt': oi, 'ids': ids, 'mask': len(ids) - 2})
    bank_scores = [0.0] * len(bank_items)
    with torch.no_grad():
        for start in range(0, len(bank_items), 32):
            batch = bank_items[start:start + 32]
            maxlen = max(len(x['ids']) for x in batch)
            ids_t = torch.full((len(batch), maxlen), PAD_ID, dtype=torch.long)
            attn_t = torch.zeros((len(batch), maxlen), dtype=torch.long)
            mpos_t = torch.zeros(len(batch), dtype=torch.long)
            for i, x in enumerate(batch):
                ids_t[i, :len(x['ids'])] = torch.tensor(x['ids']); attn_t[i, :len(x['ids'])] = 1; mpos_t[i] = x['mask']
            ids_t, attn_t = ids_t.to(device), attn_t.to(device)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                out = base(input_ids=ids_t, attention_mask=attn_t).last_hidden_state
                h = out[torch.arange(len(batch), device=device), mpos_t.to(device)]
                s = peroption_scorer(h).squeeze(-1).float()
            for i, x in enumerate(batch):
                bank_scores[start + i] = float(s[i].item())
    bank_idx = 0
    for bi, bundle in enumerate(decoupled_bundles):
        q = bundle['questions'][0]; opts = q['options']; n = len(opts)
        bundle_scores = torch.tensor(bank_scores[bank_idx:bank_idx + n]); bank_idx += n
        none_idx = opts.index('none_of_these') if 'none_of_these' in opts else -1
        real_idx = [i for i in range(n) if i != none_idx]
        real_scores = bundle_scores[real_idx]
        p_best_real = float(F.softmax(real_scores, dim=-1).max().item())
        best_real = int(real_idx[int(real_scores.argmax().item())])
        pred = none_idx if (none_idx >= 0 and p_best_real < T_THRESHOLD) else best_real
        gold = int(q['gold']); ok = int(pred == gold)
        correct_q += ok; total_q += 1; total_bundles += 1
        if ok: exact_bundles += 1
        prim_stats['choice'][0] += ok; prim_stats['choice'][1] += 1
        cell_stats[(bundle.get('source', 'unknown'), 'intent')][0] += ok
        cell_stats[(bundle.get('source', 'unknown'), 'intent')][1] += 1
        var = variants[bi] if bi < len(variants) else 'unknown'
        bank_var_stats[var][0] += ok; bank_var_stats[var][1] += 1

    overall = correct_q / max(total_q, 1)
    per_cell = {f"{s}/{q}": c / max(t, 1) for (s, q), (c, t) in sorted(cell_stats.items())}
    macro = sum(per_cell.values()) / max(len(per_cell), 1)
    print(f"\n=== UNIFIED REPRO {split_name.upper()} ===")
    print(f"Overall: {correct_q}/{total_q} = {overall:.4f} | Macro: {macro:.4f}")
    print(f"Primitives: { {k: round(c / max(t, 1), 4) for k, (c, t) in prim_stats.items()} }")
    print(f"Joint: {exact_bundles}/{total_bundles} = {exact_bundles / max(total_bundles, 1):.4f} | Contradictions: {contradicted}/{multi_q}")
    print(f"Decoupled variants: { {k: f'{c}/{t}' for k, (c, t) in sorted(bank_var_stats.items())} }")
    return {"overall_acc": overall, "macro_acc": macro, "bundle_exact_acc": exact_bundles / max(total_bundles, 1),
            "contradiction_rate": contradicted / max(multi_q, 1), "per_cell": per_cell}

repro_dev = repro_audit(dev_bundles, 'data/decision-v2-pilot/development.jsonl', "dev")
repro_test = repro_audit(test_bundles, 'data/decision-v2-pilot/test.jsonl', "test")
with open(f"{OUT_DIR}/witness-unified-repro.json", "w") as f:
    json.dump({"dev": repro_dev, "test": repro_test, "threshold_T": T_THRESHOLD, "routing": "len(options) <= 16 packed, > 16 decoupled"}, f, indent=2)
print("\nSaved witness-unified-repro.json!")
print("EXPECTED test: 1223/1440 = 0.8493 | macro 0.8502 | joint 970/1176 = 0.8248 | contradictions 0/172")


Unified label: myna-v5-unified (DeBERTa-v3-large + Cardinality-Routed Readout Head)
Routing: len(options) <= 16 -> packed_scorer | len(options) > 16 -> decoupled_scorer (T=0.40)
Packed scorer keys: ['0.bias', '0.weight', '1.bias', '1.weight', '4.bias', '4.weight']
Decoupled scorer keys: ['0.bias', '0.weight', '1.bias', '1.weight', '4.bias', '4.weight']
Using unified scorers only, threshold T=0.4
dev: packed (<=16 opts) 1060 bundles, decoupled (>16 opts) 116 bundles, variants Counter({'clean': 80, 'none_present': 12, 'none_absent': 12, 'permuted': 12})

=== UNIFIED REPRO DEV ===
Overall: 1244/1440 = 0.8639 | Macro: 0.8607
Primitives: {'noul': 0.9405, 'choice': 0.8807, 'score': 0.6542}
Joint: 995/1176 = 0.8461 | Contradictions: 1/172
Decoupled variants: {'clean': '57/80', 'none_absent': '10/12', 'none_present': '9/12', 'permuted': '9/12'}
test: packed (<=16 opts) 1060 bundles, decoupled (>16 opts) 116 bundles, variants Counter({'clean': 80, 'none_present': 12, 'none_absent': 12, 'permute